# Wedjat — Projeto completo para Google Colab

Este notebook reúne, em ordem, todas as etapas originalmente distribuídas nos notebooks 01 a 11. Ele apresenta as decisões metodológicas, executa a preparação dos dados, compara os classificadores, evolui o RAG e produz os insights comerciais finais.

## Antes de começar

1. Ative uma GPU em **Runtime → Change runtime type → GPU**.
2. Descompacte o pacote do projeto no Colab.
3. Ajuste PROJECT_DIR na próxima célula somente se a pasta estiver em outro caminho.
4. Execute as células em ordem com **Runtime → Run all**.

As transcrições são anonimizadas, mas continuam sendo dados sensíveis. Não compartilhe o runtime nem torne os arquivos públicos.

## Visão geral

1. entendimento e limpeza dos dados;
2. auditoria da base de conhecimento;
3. tokenização e chunking;
4. definição do alvo supervisionado;
5. pseudo-rotulagem e divisão agrupada;
6. baseline TF-IDF + Regressão Logística;
7. fine-tuning do BERTimbau;
8. comparação estatística dos classificadores;
9. baseline e evolução do retrieval;
10. embeddings especializados E5;
11. integração por reunião com evidências e fontes.

Os resultados supervisionados atuais ainda dependem de pseudo-rótulos. A validação humana permanece necessária antes de uso em produção.

## Preparação do ambiente

A primeira célula de código localiza a raiz do projeto, muda o diretório de trabalho e, quando necessário, instala as dependências no Colab. Essa configuração precisa ocorrer antes das etapas numeradas, pois todos os caminhos posteriores são relativos à raiz encontrada.


In [ ]:
from pathlib import Path
import os, subprocess, sys

preferred = Path('/content/Wedjat-Colab')
candidates = [preferred, Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next(
    (path for path in candidates if (path / 'requirements.txt').exists() and (path / 'data').exists()),
    None,
)
assert PROJECT_DIR is not None, (
    'Raiz do projeto não encontrada. Descompacte o pacote e ajuste o caminho preferred.'
)
os.chdir(PROJECT_DIR)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

print({'project_dir': str(PROJECT_DIR), 'google_colab': IN_COLAB})


---

# Etapa 1 de 11 — 01 data understanding

Esta seção reproduz a etapa `01_data_understanding.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 01 — Limpeza e entendimento das reuniões

Notebook autocontido para ler, validar, normalizar, deduplicar e auditar as transcrições do Wedjat.

**Objetivo e conexão com o projeto.** A entrada desta etapa são as transcrições anonimizadas em NDJSON; a saída é uma coleção de reuniões únicas, validadas e acompanhadas por estatísticas agregadas. Essa preparação estabelece a qualidade mínima necessária para que tokenização, treinamento e integração trabalhem sobre registros consistentes, sem expor o conteúdo sensível nas saídas de auditoria.


## 1. Decisões tomadas

- A reunião é a unidade de negócio e `ID_MEETING` é seu identificador.
- O NDJSON é percorrido em streaming para limitar o uso de memória.
- A limpeza normaliza somente espaços e quebras de linha; palavras, pontuação e stopwords são preservadas.
- Duplicatas exatas são removidas; IDs iguais com registros diferentes interrompem a execução.
- A saída é publicada atomicamente e nenhuma transcrição é exibida nas análises.
- O chunking será feito depois com o tokenizer do modelo e divisão agrupada por reunião.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations
import hashlib, json, os, re, tempfile
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterator, Sequence

SPEAKER_TURN_PATTERN = re.compile(r'\[LOCUTOR\s+\d+\]:')


### Funções auxiliares da seção

Esta célula agrupa `DataPreparationError`, `PreparationSummary`, `iter_ndjson`, `normalize_transcript`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
class DataPreparationError(ValueError):
    pass

@dataclass(frozen=True)
class PreparationSummary:
    input_records: int
    output_meetings: int
    exact_duplicates_removed: int
    total_transcript_characters: int
    total_speaker_turns: int
    shortest_transcript_characters: int
    longest_transcript_characters: int

def iter_ndjson(path: Path) -> Iterator[tuple[int, dict[str, Any]]]:
    with path.open('r', encoding='utf-8') as source:
        for line_number, raw_line in enumerate(source, 1):
            if not raw_line.strip():
                continue
            try:
                record = json.loads(raw_line)
            except json.JSONDecodeError as exc:
                raise DataPreparationError(f'JSON inválido na linha {line_number}: coluna {exc.colno}.') from exc
            if not isinstance(record, dict):
                raise DataPreparationError(f'A linha {line_number} deve conter um objeto JSON.')
            yield line_number, record

def normalize_transcript(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in text.split('\n')]
    return '\n'.join(line for line in lines if line)


### Funções auxiliares da seção

Esta célula agrupa `validate_and_enrich_record`, `record_fingerprint`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def validate_and_enrich_record(record, line_number, required_fields):
    for field in required_fields:
        if field not in record or record[field] in (None, ''):
            raise DataPreparationError(f'Campo obrigatório {field!r} ausente na linha {line_number}.')
    meeting_id = str(record['ID_MEETING']).strip()
    transcript = record['ANON_TRANSCRICAO']
    if not meeting_id:
        raise DataPreparationError(f'ID_MEETING vazio na linha {line_number}.')
    if not isinstance(transcript, str):
        raise DataPreparationError(f'ANON_TRANSCRICAO deve ser texto na linha {line_number}.')
    transcript = normalize_transcript(transcript)
    if not transcript:
        raise DataPreparationError(f'ANON_TRANSCRICAO vazia após normalização na linha {line_number}.')
    prepared = dict(record)
    prepared.update({
        'ID_MEETING': meeting_id,
        'ANON_TRANSCRICAO': transcript,
        'NUM_CARACTERES_TRANSCRICAO': len(transcript),
        'NUM_TURNOS_TRANSCRICAO': len(SPEAKER_TURN_PATTERN.findall(transcript)),
    })
    return prepared

def record_fingerprint(record):
    canonical = json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()


### Função auxiliar: `write_json_atomic`

Esta célula isola a responsabilidade implementada por `write_json_atomic`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def write_json_atomic(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, prefix=f'.{path.name}.', suffix='.tmp', delete=False) as handle:
        json.dump(content, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
        temporary_path = Path(handle.name)
    os.replace(temporary_path, path)


### Preparação e deduplicação das reuniões

A rotina coordena validação, deduplicação e escrita atômica. Duplicatas exatas são removidas, enquanto IDs iguais com conteúdos diferentes são tratados como conflito.


In [ ]:
def prepare_meetings(input_path, output_path, report_path, required_fields):
    if not input_path.is_file():
        raise DataPreparationError(f'Arquivo de entrada não encontrado: {input_path}')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fingerprints = {}
    input_records = output_meetings = duplicates = total_characters = total_turns = 0
    shortest = None
    longest = 0
    temporary_handle = tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=output_path.parent, prefix=f'.{output_path.name}.', suffix='.tmp', delete=False)
    temporary_path = Path(temporary_handle.name)
    try:
        with temporary_handle as destination:
            for line_number, source_record in iter_ndjson(input_path):
                input_records += 1
                prepared = validate_and_enrich_record(source_record, line_number, required_fields)
                meeting_id = prepared['ID_MEETING']
                fingerprint = record_fingerprint(prepared)
                previous = fingerprints.get(meeting_id)
                if previous is not None:
                    if previous == fingerprint:
                        duplicates += 1
                        continue
                    raise DataPreparationError(f'ID_MEETING conflitante na linha {line_number}.')
                fingerprints[meeting_id] = fingerprint
                destination.write(json.dumps(prepared, ensure_ascii=False) + '\n')
                characters = prepared['NUM_CARACTERES_TRANSCRICAO']
                output_meetings += 1
                total_characters += characters
                total_turns += prepared['NUM_TURNOS_TRANSCRICAO']
                shortest = characters if shortest is None else min(shortest, characters)
                longest = max(longest, characters)
        os.replace(temporary_path, output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise
    summary = PreparationSummary(input_records, output_meetings, duplicates, total_characters, total_turns, shortest or 0, longest)
    write_json_atomic(report_path, asdict(summary))
    return summary


## 3. Execução

A saída processada continua fora do Git porque contém transcrições. O relatório contém apenas totais seguros.

A execução aplica as funções definidas anteriormente aos caminhos configurados. A escrita protegida evita publicar uma saída parcial caso alguma validação interrompa o processamento.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_PATH = PROJECT_ROOT / 'data' / 'raw' / 'ANON_transcricao (2).json'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'meetings.jsonl'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'metrics' / 'data_preparation_summary.json'
summary = prepare_meetings(INPUT_PATH, OUTPUT_PATH, REPORT_PATH, ['ID_MEETING', 'ANON_TRANSCRICAO'])
asdict(summary)

## 4. Auditoria segura

Exibimos o esquema sem valores e estatísticas de tamanho para orientar o próximo chunking.

A auditoria usa apenas estrutura e valores agregados. Essa escolha permite avaliar cobertura e distribuição sem imprimir transcrições ou outros conteúdos que não devem aparecer no material entregue.


In [ ]:
_, first_record = next(iter_ndjson(INPUT_PATH))
{field: {'type': type(value).__name__, 'is_null': value is None, 'string_length': len(value) if isinstance(value, str) else None} for field, value in first_record.items()}

### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
character_counts, speaker_turn_counts = [], []
for _, record in iter_ndjson(OUTPUT_PATH):
    character_counts.append(record['NUM_CARACTERES_TRANSCRICAO'])
    speaker_turn_counts.append(record['NUM_TURNOS_TRANSCRICAO'])


### Função auxiliar: `percentile`

Esta célula isola a responsabilidade implementada por `percentile`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def percentile(values, probability):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * probability)]


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
{'meetings': len(character_counts), 'characters_mean': round(mean(character_counts), 2), 'characters_median': median(character_counts), 'characters_p90': percentile(character_counts, .90), 'characters_p95': percentile(character_counts, .95), 'speaker_turns_mean': round(mean(speaker_turn_counts), 2), 'speaker_turns_p95': percentile(speaker_turn_counts, .95)}


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 2 de 11 — 02 rag knowledge base

Esta seção reproduz a etapa `02_rag_knowledge_base.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 02 — Exploração da base TOTVS para RAG

A base já está entregue como JSON em `data/knowledge_base/totvs_rag_kb_v1.json`. Este notebook apenas carrega, valida e explora o artefato pronto.

**Objetivo e conexão com o projeto.** Esta etapa examina a base TOTVS que dará contexto aos resultados do classificador e verifica se cada documento possui o contrato mínimo esperado pelo retrieval. A busca lexical apresentada aqui funciona como referência inicial: ela permite validar conteúdo, metadados e fontes antes de introduzir métodos semânticos mais complexos.


## 1. Decisões tomadas

- O JSON contém uma lista de 107 chunks independentes e não depende de um conversor no projeto.
- Cada chunk mantém ID, tipo, produto, categoria, listas de metadados, nível de evidência, conteúdo, fontes e data de compilação.
- O nível de evidência será usado futuramente para impedir que hipóteses comerciais sejam apresentadas como fatos.
- A busca lexical abaixo é apenas um baseline transparente; embeddings e reranking serão comparados em uma próxima etapa.
- Resultados mostram metadados e fontes, evitando despejar todo o conteúdo na saída do notebook.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from collections import Counter
from pathlib import Path
import json, re, unicodedata

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
KB_PATH = PROJECT_ROOT / 'data' / 'knowledge_base' / 'totvs_rag_kb_v1.json'
records = json.loads(KB_PATH.read_text(encoding='utf-8'))
len(records)

## 3. Validação e auditoria

Validamos o contrato mínimo, unicidade dos IDs e presença de fontes antes de usar a base em recuperação.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
required_fields = {'id', 'title', 'document_type', 'company', 'product', 'category', 'evidence_level', 'content', 'sources', 'compiled_at'}
assert isinstance(records, list) and records
assert all(required_fields <= record.keys() for record in records)
assert len({record['id'] for record in records}) == len(records)
assert all(record['sources'] for record in records)
{
    'total_chunks': len(records),
    'unique_ids': len({record['id'] for record in records}),
    'document_types': dict(Counter(record['document_type'] for record in records)),
    'evidence_levels': dict(Counter(record['evidence_level'] for record in records)),
    'unique_source_urls': len({source['url'] for record in records for source in record['sources']}),
}

## 4. Baseline de busca lexical

Esta função mede sobreposição de termos em título, conteúdo e palavras-chave. Ela servirá como referência simples para avaliar se embeddings realmente melhoram a recuperação.


In [ ]:
def normalize_for_search(text):
    text = unicodedata.normalize('NFKD', text.casefold())
    return ''.join(char for char in text if not unicodedata.combining(char))

def lexical_search(query, top_k=5):
    query_terms = set(re.findall(r'\w+', normalize_for_search(query)))
    ranked = []
    for record in records:
        searchable = ' '.join((record['title'], record['content'], ' '.join(record['keywords'])))
        terms = set(re.findall(r'\w+', normalize_for_search(searchable)))
        score = len(query_terms & terms)
        if score:
            ranked.append((score, record))
    ranked.sort(key=lambda item: (-item[0], item[1]['id']))
    return [
        {'score': score, 'id': record['id'], 'title': record['title'], 'evidence_level': record['evidence_level'], 'source_urls': [source['url'] for source in record['sources']]}
        for score, record in ranked[:top_k]
    ]


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
lexical_search('estoque divergente entre filiais e separação no armazém')


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 3 de 11 — 03 tokenization and chunking

Esta seção reproduz a etapa `03_tokenization_and_chunking.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 03 — Tokenização e chunking com BERTimbau

Este notebook transforma as reuniões limpas em chunks compatíveis com o BERTimbau, preservando reunião, ordem, locutores e posições no texto original.

**Objetivo e conexão com o projeto.** A entrada são as reuniões limpas; a saída são janelas ordenadas que respeitam o limite real de tokens do BERTimbau e mantêm a rastreabilidade até a reunião original. O chunk é a unidade técnica de classificação, enquanto a reunião continua sendo a unidade de negócio. Preservar essa distinção evita que limitações do modelo alterem o significado da análise.


## 1. Decisões tomadas

- Usamos `neuralmind/bert-base-portuguese-cased`, o BERTimbau Base para português brasileiro.
- O modelo admite 512 posições. Reservamos 2 para `[CLS]` e `[SEP]`, deixando 510 tokens de conteúdo.
- A sobreposição-alvo é de 64 tokens e reutiliza palavras completas.
- Os limites são definidos pelos tokens reais do tokenizer, mas cada chunk começa e termina em fronteira de palavra.
- O texto do chunk é recortado diretamente da transcrição normalizada; não usamos `decode`, evitando alterações artificiais no texto.
- Cada chunk mantém `meeting_id`, índice, offsets de caracteres, intervalo de turnos e locutores.
- Nenhuma transcrição ou trecho é exibido nas saídas do notebook.
- Os chunks permanecem em `data/processed/`, fora do Git. Futuras divisões de treino/teste serão agrupadas por reunião.

Referências: [BERTimbau Base](https://huggingface.co/neuralmind/bert-base-portuguese-cased) e [API de tokenização](https://huggingface.co/docs/transformers/main_classes/tokenizer).


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json, os, re, tempfile
from collections import Counter
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterator

import transformers
from transformers import AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'meetings.jsonl'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_bertimbau.jsonl'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'metrics' / 'chunking_summary.json'
CHECKPOINT = 'neuralmind/bert-base-portuguese-cased'
MAX_SEQUENCE_LENGTH = 512
CONTENT_MAX_TOKENS = 510
OVERLAP_TOKENS = 64
SPEAKER_PATTERN = re.compile(r'\[LOCUTOR\s+(\d+)\]:')

## 3. Carregar e conferir o tokenizer

Baixamos apenas os arquivos do tokenizer. O modelo neural completo será necessário somente no fine-tuning.

A célula seguinte reúne somente a leitura e a conferência dos insumos desta etapa. Validar caminhos, formato e contrato antes das transformações torna falhas de entrada explícitas e evita resultados parciais difíceis de diagnosticar.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT, use_fast=True, do_lower_case=False
)
special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
assert special_tokens == 2
assert CONTENT_MAX_TOKENS + special_tokens <= MAX_SEQUENCE_LENGTH
{
    'checkpoint': CHECKPOINT,
    'transformers_version': transformers.__version__,
    'tokenizer_class': type(tokenizer).__name__,
    'is_fast': tokenizer.is_fast,
    'special_tokens': special_tokens,
    'content_max_tokens': CONTENT_MAX_TOKENS,
    'overlap_tokens_target': OVERLAP_TOKENS,
}

## 4. Funções de turnos e janelas de palavras

Os offsets retornados pelo tokenizer são associados às palavras e aos turnos no texto original. A sobreposição repete apenas contexto; ela nunca cria mistura entre reuniões.


In [ ]:
def iter_ndjson(path: Path) -> Iterator[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as source:
        for line_number, line in enumerate(source, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'Registro inválido na linha {line_number}.')
            yield record

def speaker_spans(text: str) -> tuple[list[dict[str, Any]], bool]:
    matches = list(SPEAKER_PATTERN.finditer(text))
    spans = []
    has_preamble = bool(matches and text[:matches[0].start()].strip())
    if not matches:
        return ([{'turn_index': 0, 'speaker': None, 'char_start': 0, 'char_end': len(text)}], False)
    if has_preamble:
        spans.append({'turn_index': 0, 'speaker': None, 'char_start': 0, 'char_end': matches[0].start()})
    index_offset = len(spans)
    for index, match in enumerate(matches):
        spans.append({
            'turn_index': index + index_offset,
            'speaker': f'LOCUTOR {match.group(1)}',
            'char_start': match.start(),
            'char_end': matches[index + 1].start() if index + 1 < len(matches) else len(text),
        })
    return spans, has_preamble


### Funções auxiliares da seção

Esta célula agrupa `word_spans_with_token_counts`, `intersecting_turns`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def word_spans_with_token_counts(text: str) -> tuple[list[tuple[int, int]], list[int], int]:
    words = [(match.start(), match.end()) for match in re.finditer(r'\S+', text)]
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = encoded['offset_mapping']
    counts = []
    token_index = 0
    for start, end in words:
        while token_index < len(offsets) and offsets[token_index][1] <= start:
            token_index += 1
        cursor = token_index
        count = 0
        while cursor < len(offsets) and offsets[cursor][0] < end:
            if offsets[cursor][1] > start:
                count += 1
            cursor += 1
        counts.append(count)
        token_index = cursor
    return words, counts, len(encoded['input_ids'])

def intersecting_turns(turns, char_start, char_end):
    return [turn for turn in turns if turn['char_end'] > char_start and turn['char_start'] < char_end]


### Construção dos chunks de uma reunião

A função cria janelas dentro do limite do BERTimbau, aplica a sobreposição-alvo e preserva ordem, offsets, turnos e locutores para cada chunk.


In [ ]:
def chunk_meeting(record: dict[str, Any]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    meeting_id = str(record['ID_MEETING'])
    text = record['ANON_TRANSCRICAO']
    turns, has_preamble = speaker_spans(text)
    words, word_token_counts, meeting_tokens = word_spans_with_token_counts(text)
    turn_token_counts = [0] * len(turns)
    turn_cursor = 0
    for (word_start, _), token_count in zip(words, word_token_counts):
        while turn_cursor + 1 < len(turns) and word_start >= turns[turn_cursor]['char_end']:
            turn_cursor += 1
        turn_token_counts[turn_cursor] += token_count
    if not words:
        raise ValueError(f'Reunião {meeting_id!r} sem palavras.')
    if max(word_token_counts, default=0) > CONTENT_MAX_TOKENS:
        raise ValueError(f'Reunião {meeting_id!r} contém palavra maior que o limite do modelo.')

    chunks = []
    start_word = 0
    while start_word < len(words):
        end_word = start_word
        estimated_tokens = 0
        while end_word < len(words) and estimated_tokens + word_token_counts[end_word] <= CONTENT_MAX_TOKENS:
            estimated_tokens += word_token_counts[end_word]
            end_word += 1
        if end_word == start_word:
            end_word += 1
        char_start = words[start_word][0]
        char_end = words[end_word - 1][1]
        chunk_text = text[char_start:char_end]
        actual_tokens = len(tokenizer.encode(chunk_text, add_special_tokens=False))
        while actual_tokens > CONTENT_MAX_TOKENS and end_word > start_word + 1:
            end_word -= 1
            char_end = words[end_word - 1][1]
            chunk_text = text[char_start:char_end]
            actual_tokens = len(tokenizer.encode(chunk_text, add_special_tokens=False))
        chunk_turns = intersecting_turns(turns, char_start, char_end)
        chunk_index = len(chunks)
        chunks.append({
            'meeting_id': meeting_id,
            'chunk_id': f'{meeting_id}::chunk-{chunk_index:04d}',
            'chunk_index': chunk_index,
            'text': chunk_text,
            'num_tokens': actual_tokens,
            'char_start': char_start,
            'char_end': char_end,
            'turn_start': chunk_turns[0]['turn_index'],
            'turn_end': chunk_turns[-1]['turn_index'],
            'speakers': sorted({turn['speaker'] for turn in chunk_turns if turn['speaker']}),
            'checkpoint': CHECKPOINT,
            'content_max_tokens': CONTENT_MAX_TOKENS,
            'overlap_tokens_target': OVERLAP_TOKENS,
        })
        if end_word >= len(words):
            break
        overlap_start = end_word
        overlap_count = 0
        while overlap_start > start_word and overlap_count + word_token_counts[overlap_start - 1] <= OVERLAP_TOKENS:
            overlap_start -= 1
            overlap_count += word_token_counts[overlap_start]
        start_word = overlap_start if overlap_start > start_word else end_word

    covered_turns = {index for chunk in chunks for index in range(chunk['turn_start'], chunk['turn_end'] + 1)}
    assert len(covered_turns) == len(turns)
    assert all(0 < chunk['num_tokens'] <= CONTENT_MAX_TOKENS for chunk in chunks)
    assert [chunk['chunk_index'] for chunk in chunks] == list(range(len(chunks)))
    return chunks, {
        'meeting_tokens': meeting_tokens,
        'speaker_turns': len(turns),
        'turn_token_counts': turn_token_counts,
        'has_speaker_marker': bool(SPEAKER_PATTERN.search(text)),
        'has_preamble': has_preamble,
    }


## 5. Gerar chunks e relatório

A escrita é atômica. Se alguma reunião falhar nas validações, o arquivo anterior não é substituído por uma saída parcial.

O processamento em chunks resolve o limite de contexto sem perder a ligação com a reunião. Índices, offsets e sobreposição permitem reconstruir a ordem das evidências e agregar previsões posteriormente.


In [ ]:
def percentile(values, probability):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * probability)]

def write_json_atomic(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, prefix=f'.{path.name}.', suffix='.tmp', delete=False) as handle:
        json.dump(content, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
        temporary_path = Path(handle.name)
    os.replace(temporary_path, path)


### Leitura ou escrita dos artefatos

A operação de arquivo fica isolada nesta célula para tornar claro quais dados entram ou saem da etapa e em que momento as validações são aplicadas.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_handle = tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=OUTPUT_PATH.parent, prefix=f'.{OUTPUT_PATH.name}.', suffix='.tmp', delete=False)
temporary_path = Path(temporary_handle.name)
meeting_token_counts, turn_token_counts, chunks_per_meeting, chunk_token_counts = [], [], [], []
meetings_without_marker = meetings_with_preamble = total_meetings = total_chunks = 0
try:
    with temporary_handle as destination:
        for meeting in iter_ndjson(INPUT_PATH):
            chunks, audit = chunk_meeting(meeting)
            total_meetings += 1
            total_chunks += len(chunks)
            meeting_token_counts.append(audit['meeting_tokens'])
            turn_token_counts.extend(audit['turn_token_counts'])
            chunks_per_meeting.append(len(chunks))
            chunk_token_counts.extend(chunk['num_tokens'] for chunk in chunks)
            meetings_without_marker += not audit['has_speaker_marker']
            meetings_with_preamble += audit['has_preamble']
            metadata = {key: value for key, value in meeting.items() if key != 'ANON_TRANSCRICAO'}
            for chunk in chunks:
                destination.write(json.dumps({**metadata, **chunk}, ensure_ascii=False) + '\n')
    os.replace(temporary_path, OUTPUT_PATH)
except Exception:
    temporary_path.unlink(missing_ok=True)
    raise


### Consolidação do relatório da etapa

Configurações, contagens, métricas e alertas são reunidos em um resumo auditável. O relatório registra resultados agregados sem acrescentar novas transformações aos dados.


In [ ]:
summary = {
    'checkpoint': CHECKPOINT,
    'transformers_version': transformers.__version__,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'content_max_tokens': CONTENT_MAX_TOKENS,
    'overlap_tokens_target': OVERLAP_TOKENS,
    'input_meetings': total_meetings,
    'output_chunks': total_chunks,
    'meetings_without_speaker_marker': meetings_without_marker,
    'meetings_with_text_before_first_marker': meetings_with_preamble,
    'meeting_tokens_mean': round(mean(meeting_token_counts), 2),
    'meeting_tokens_median': median(meeting_token_counts),
    'meeting_tokens_p95': percentile(meeting_token_counts, 0.95),
    'turn_tokens_mean': round(mean(turn_token_counts), 2),
    'turn_tokens_median': median(turn_token_counts),
    'turn_tokens_p95': percentile(turn_token_counts, 0.95),
    'turn_tokens_max': max(turn_token_counts),
    'chunks_per_meeting_mean': round(mean(chunks_per_meeting), 2),
    'chunks_per_meeting_p95': percentile(chunks_per_meeting, 0.95),
    'chunks_per_meeting_max': max(chunks_per_meeting),
    'chunk_tokens_mean': round(mean(chunk_token_counts), 2),
    'chunk_tokens_max': max(chunk_token_counts),
}
write_json_atomic(REPORT_PATH, summary)
summary


## 6. Validação final sem conteúdo sensível

Relemos o arquivo e confirmamos IDs únicos, limites de tokens, ordem dos chunks e agrupamento contíguo por reunião.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
seen_chunk_ids = set()
last_index_by_meeting = {}
observed_meetings = set()
validated_chunks = 0
for chunk in iter_ndjson(OUTPUT_PATH):
    assert chunk['chunk_id'] not in seen_chunk_ids
    assert 0 < chunk['num_tokens'] <= CONTENT_MAX_TOKENS
    expected_index = last_index_by_meeting.get(chunk['meeting_id'], -1) + 1
    assert chunk['chunk_index'] == expected_index
    seen_chunk_ids.add(chunk['chunk_id'])
    last_index_by_meeting[chunk['meeting_id']] = chunk['chunk_index']
    observed_meetings.add(chunk['meeting_id'])
    validated_chunks += 1

assert validated_chunks == summary['output_chunks']
assert len(observed_meetings) == summary['input_meetings']
{'validated_chunks': validated_chunks, 'validated_meetings': len(observed_meetings), 'unique_chunk_ids': len(seen_chunk_ids), 'max_tokens_observed': summary['chunk_tokens_max']}

### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 4 de 11 — 04 target and bertimbau

Esta seção reproduz a etapa `04_target_and_bertimbau.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 04 — Alvo supervisionado e funcionamento do BERTimbau

Este notebook traduz os requisitos da Sprint 3 para um experimento concreto e mostra como o BERTimbau receberá os trechos das reuniões.

**Objetivo e conexão com o projeto.** Antes do treinamento, esta etapa formaliza o alvo binário, a unidade de anotação e o formato que o BERTimbau receberá. O objetivo é tornar explícito o que conta como oportunidade e reconhecer que os dados ainda não trazem verdade-terreno humana; por isso, a configuração do modelo não é apresentada como um classificador já treinado.


## 1. Contrato da Sprint 3

A Sprint permite diferentes problemas de classificação. Para manter um experimento claro e alinhado à base comercial TOTVS, adotamos inicialmente:

- **Problema:** detecção de oportunidade comercial;
- **Classes:** `nao_oportunidade` e `oportunidade`;
- **Unidade rotulada:** chunk de transcrição;
- **Unidade de separação:** reunião (`meeting_id`);
- **Agregação futura:** maior probabilidade de oportunidade entre os chunks da reunião, com limiar validado;
- **Algoritmo 1:** TF-IDF + Logistic Regression;
- **Algoritmo 2:** BERTimbau fine-tuned para classificação binária;
- **Métricas:** confusion matrix, precision, recall, F1-score e accuracy;
- **Métrica principal proposta:** recall da classe `oportunidade`, acompanhado de precision e F1. Perder uma oportunidade real pode ser mais caro do que revisar um alerta falso, mas o limiar deverá controlar excesso de alertas.

Os dois modelos usarão os mesmos rótulos e o mesmo conjunto de teste agrupado por reunião.


## 2. Guia inicial de anotação

### `oportunidade`
Marcar quando o trecho trouxer evidência comercial acionável, por exemplo: dor que pode ser atendida, necessidade explícita, intenção de compra, avaliação ou troca de fornecedor, pedido de proposta/demonstração, orçamento, prazo de decisão, expansão, cross-sell ou upsell.

### `nao_oportunidade`
Marcar quando o trecho for contextual, operacional ou social sem necessidade comercial acionável: apresentação, confirmação de agenda, suporte rotineiro, descrição neutra do ambiente ou menção de produto sem dor/intenção.

### `revisao`
Usar somente no processo de anotação quando faltar contexto ou houver dúvida. Esses exemplos não entram no treino até revisão humana. `revisao` não é uma terceira classe do modelo.

A base RAG pode ajudar o anotador a compreender produtos e dores, mas nunca deve gerar o rótulo de verdade automaticamente.


## 3. Como o BERTimbau será usado

O BERTimbau disponível hoje é um modelo de linguagem pré-treinado, não um classificador comercial pronto. O fine-tuning adiciona uma cabeça de classificação sobre a representação do token `[CLS]` e ajusta os pesos usando nossos chunks rotulados.

Fluxo de um chunk:

1. o tokenizer converte texto em IDs de subpalavras;
2. adiciona `[CLS]` no início e `[SEP]` no fim;
3. aplica padding e attention mask até o tamanho configurado;
4. o BERTimbau produz uma representação contextual;
5. a cabeça de classificação retorna logits para as duas classes;
6. softmax converte logits em probabilidades;
7. o limiar de decisão é escolhido na validação, nunca no teste.

Sem rótulos humanos, executar o modelo pré-treinado permitiria apenas tarefas como preencher palavras mascaradas; isso não mede oportunidade comercial.


## 4. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from pathlib import Path
import json

import transformers
from transformers import AutoConfig, AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CHUNKS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_bertimbau.jsonl'
CHECKPOINT = 'neuralmind/bert-base-portuguese-cased'
LABEL2ID = {'nao_oportunidade': 0, 'oportunidade': 1}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

## 5. Verificar o contrato atual dos dados

A inspeção mostra apenas os nomes dos campos. Nenhum conteúdo de transcrição é exibido.

Antes de transformar ou modelar, conferimos os dados usados nesta etapa e as condições que garantem comparabilidade. As verificações protegem a sequência do notebook contra arquivos incompletos e partições incompatíveis.


In [ ]:
with CHUNKS_PATH.open('r', encoding='utf-8') as source:
    first_chunk = json.loads(next(source))
fields = sorted(first_chunk)
label_like_fields = [field for field in fields if any(term in field.casefold() for term in ('label', 'rotulo', 'rótulo', 'target', 'classe'))]
{'available_fields': fields, 'existing_label_fields': label_like_fields}

## 6. Demonstração segura do tokenizer

Usamos frases sintéticas para visualizar as entradas. O mesmo processo será aplicado aos chunks reais sem exibi-los.

A tokenização traduz o texto para a representação aceita pelo modelo. Conferir tokens especiais, limites e máscaras é necessário para que cada entrada respeite a arquitetura e para que o chunking não dependa de uma estimativa por caracteres.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=True, do_lower_case=False)
examples = [
    'Precisamos automatizar as aprovações e gostaríamos de avaliar uma solução.',
    'A reunião de acompanhamento ficou marcada para terça-feira.',
]
encoded = tokenizer(examples, padding=True, truncation=True, max_length=512)
[
    {
        'synthetic_text': text,
        'tokens': tokenizer.convert_ids_to_tokens(input_ids),
        'attention_tokens': sum(mask),
    }
    for text, input_ids, mask in zip(examples, encoded['input_ids'], encoded['attention_mask'])
]

## 7. Configuração futura do classificador

Esta célula prepara a configuração da cabeça binária sem baixar os pesos do modelo. O fine-tuning exigirá PyTorch e o conjunto anotado.


In [ ]:
config = AutoConfig.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABEL2ID),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)
{
    'model_type': config.model_type,
    'hidden_size': config.hidden_size,
    'layers': config.num_hidden_layers,
    'attention_heads': config.num_attention_heads,
    'max_position_embeddings': config.max_position_embeddings,
    'labels': config.id2label,
    'transformers_version': transformers.__version__,
}

## 8. Próxima etapa necessária

Antes de treinar, precisamos criar uma fila de anotação e obter rótulos humanos. Depois construiremos uma única divisão por `meeting_id`, treinaremos Logistic Regression e BERTimbau nela e compararemos as métricas no mesmo teste.

Os itens a seguir separam o que já foi demonstrado do que ainda depende de dados humanos, calibração ou evolução técnica. Assim, o notebook termina sem transformar resultados experimentais em garantias de produção.


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 5 de 11 — 05 weak supervision and split

Esta seção reproduz a etapa `05_weak_supervision_and_split.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 05 — Supervisão fraca, pseudo-rótulos e divisão segura

Este notebook cria a primeira versão dos rótulos de **oportunidade comercial** sem fingir que previsões automáticas são verdade-terreno. Ele combina dois rotuladores independentes:

1. regras comerciais apoiadas pelo vocabulário do RAG;
2. similaridade semântica com protótipos usando BERTimbau.

Somente concordâncias nos extremos semânticos são aceitas como pseudo-rótulos. Conflitos e uma amostra das concordâncias formam uma fila de auditoria humana. **Precisão, recall e F1 reais só poderão ser calculados depois da anotação humana.**

**Objetivo e conexão com o projeto.** Esta etapa cria dados de desenvolvimento provisórios ao combinar regras comerciais e similaridade semântica, mantendo separadas concordâncias automáticas, conflitos e itens destinados à revisão humana. A divisão é realizada por reunião para impedir que trechos do mesmo encontro apareçam em partições diferentes. Esse cuidado é essencial porque chunks vizinhos compartilham contexto e poderiam inflar a validação.


## 1. Decisões de execução

- O dispositivo é detectado automaticamente: RTX local, GPU do Colab ou CPU.
- O primeiro experimento usa 3.000 chunks, selecionados com semente fixa e cobertura inicial de todas as reuniões.
- O BERTimbau recebe no máximo 512 tokens, igual ao limite de sua arquitetura.
- Na RTX 3050 de 8 GB usamos batch 8 e inferência em precisão mista; na CPU usamos batch 2.
- A divisão de desenvolvimento é agrupada por `meeting_id`; nunca separamos chunks da mesma reunião.
- Ainda não criamos um conjunto de teste automático: o teste final deve conter apenas rótulos humanos.
- Nenhum trecho de transcrição é impresso nas saídas do notebook.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json
import random
import re
import time
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
INITIAL_SAMPLE_SIZE = 3_000
AUDIT_SIZE = 150
MAX_LENGTH = 512

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2


### Função auxiliar: `find_project_root`

Esta célula isola a responsabilidade implementada por `find_project_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada. No Colab, entre na pasta Wedjat antes de executar.")


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `CHUNKS_PATH`, `KB_PATH`, `PSEUDO_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT = find_project_root()
CHUNKS_PATH = ROOT / "data/processed/chunks_bertimbau.jsonl"
KB_PATH = ROOT / "data/knowledge_base/totvs_rag_kb_v1.json"
PSEUDO_PATH = ROOT / "data/processed/pseudo_labels_opportunity.jsonl"
AUDIT_PATH = ROOT / "data/processed/annotation_queue_opportunity.jsonl"
SUMMARY_PATH = ROOT / "reports/metrics/weak_supervision_summary.json"

device_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print({"device": str(DEVICE), "device_name": device_name, "batch_size": BATCH_SIZE})


## 3. Amostra reproduzível

Primeiro escolhemos um chunk aleatório por reunião. As vagas restantes são preenchidas aleatoriamente entre os demais chunks. Isso evita que as reuniões mais longas dominem toda a primeira rodada, sem impedir que algumas contribuam com mais de um exemplo.

A seleção reproduzível controla quais exemplos entram no experimento e mantém cobertura por reunião. A semente fixa permite repetir a mesma análise e separar mudanças metodológicas de variações aleatórias da amostra.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


### Preparação dos objetos desta etapa

A célula prepara `all_chunks`, `by_meeting`, `rng`, `sample` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
all_chunks = read_jsonl(CHUNKS_PATH)
by_meeting: dict[str, list[dict]] = defaultdict(list)
for row in all_chunks:
    by_meeting[str(row["meeting_id"])].append(row)

rng = random.Random(SEED)
sample = [rng.choice(rows) for _, rows in sorted(by_meeting.items())]
selected_ids = {row["chunk_id"] for row in sample}
remaining = [row for row in all_chunks if row["chunk_id"] not in selected_ids]
target_size = min(INITIAL_SAMPLE_SIZE, len(all_chunks))
if len(sample) < target_size:
    sample.extend(rng.sample(remaining, target_size - len(sample)))
sample.sort(key=lambda row: row["chunk_id"])

print({
    "total_chunks": len(all_chunks),
    "sample_chunks": len(sample),
    "meetings_in_sample": len({row["meeting_id"] for row in sample}),
})


## 4. Rotulador 1 — regras comerciais apoiadas pelo RAG

Os chunks do RAG dos tipos `sinal_oportunidade` e `dor_para_produto` fornecem vocabulário de produtos e dores. Esse vocabulário só conta como contexto: para marcar oportunidade positiva, também exigimos intenção, compra, implantação ou uma dor explícita. Regras incertas se abstêm (`-1`).

Este rotulador produz apenas um sinal provisório. Os casos de abstenção e conflito são preservados porque forçar uma classe nesses exemplos aumentaria o ruído e daria uma falsa impressão de verdade-terreno.


In [ ]:
def normalize_for_match(text: str) -> str:
    text = unicodedata.normalize("NFKD", text.lower())
    return "".join(char for char in text if not unicodedata.combining(char))


### Preparação dos objetos desta etapa

A célula prepara `knowledge_base`, `rag_terms`, `PRODUCT_PATTERN`, `INTENT_PATTERN` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
knowledge_base = json.loads(KB_PATH.read_text(encoding="utf-8"))
rag_terms: set[str] = set()
for document in knowledge_base:
    if document.get("document_type") in {"sinal_oportunidade", "dor_para_produto", "produto"}:
        candidates = [document.get("product", ""), *document.get("keywords", [])]
        rag_terms.update(normalize_for_match(term) for term in candidates if len(str(term).strip()) >= 4)

PRODUCT_PATTERN = re.compile(
    r"\b(?:erp|crm|software|sistema|solucao|plataforma|protheus|fluig|datasul|rm|totvs|folha|hcm|wms|licenca|modulo)\b"
)
INTENT_PATTERN = re.compile(
    r"\b(?:precisamos?|necessitamos?|queremos?|gostariamos?|buscamos?|procuramos?|avaliar|avaliando|interesse|interessados?)\b"
)
BUY_PATTERN = re.compile(
    r"\b(?:proposta|cotacao|orcamento|preco|valor|contratar|adquirir|comprar|implantacao|implementar|migrar|substituir|prazo)\b"
)
PAIN_PATTERN = re.compile(
    r"\b(?:problema|dificuldade|gargalo|retrabalho|manual|planilha|integracao|lentidao|erro|nao atende|limitacao)\b"
)
REFUSAL_PATTERN = re.compile(
    r"\b(?:sem interesse|nao temos interesse|nao precisamos|nao pretendemos|nao vamos contratar|projeto cancelado|projeto descartado)\b"
)


### Funções auxiliares da seção

Esta célula agrupa `rag_context_hits`, `rule_label`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def rag_context_hits(normalized_text: str) -> list[str]:
    return sorted(term for term in rag_terms if len(term) >= 5 and term in normalized_text)[:20]

def rule_label(text: str) -> tuple[int, int, list[str]]:
    normalized = normalize_for_match(text)
    if REFUSAL_PATTERN.search(normalized):
        return 0, 4, ["recusa_explicita"]

    signals = {
        "intencao": bool(INTENT_PATTERN.search(normalized)),
        "compra_implantacao": bool(BUY_PATTERN.search(normalized)),
        "dor": bool(PAIN_PATTERN.search(normalized)),
        "produto": bool(PRODUCT_PATTERN.search(normalized)),
        "contexto_rag": bool(rag_context_hits(normalized)),
    }
    score = 2 * signals["intencao"] + 2 * signals["compra_implantacao"] + signals["dor"] + signals["produto"] + signals["contexto_rag"]
    reasons = [name for name, present in signals.items() if present]

    if score >= 4 and (signals["intencao"] or signals["compra_implantacao"]):
        return 1, score, reasons
    if not (signals["intencao"] or signals["compra_implantacao"] or signals["dor"]):
        return 0, score, ["sem_intencao_compra_ou_dor"]
    return -1, score, reasons


### Preparação dos objetos desta etapa

A célula prepara `rule_results` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
rule_results = [rule_label(row["text"]) for row in sample]
print({"rule_labels": dict(Counter(label for label, _, _ in rule_results))})


## 5. Rotulador 2 — BERTimbau por protótipos

O BERTimbau base ainda não é um classificador de oportunidades. Aqui ele transforma textos e exemplos prototípicos em vetores; a decisão usa a diferença entre a similaridade com protótipos positivos e negativos. É um sinal semântico inicial, não o fine-tuning final da Sprint 3. Para reduzir ruído, apenas os 30% mais negativos e os 30% mais positivos recebem rótulo; a faixa central se abstém.

Este rotulador produz apenas um sinal provisório. Os casos de abstenção e conflito são preservados porque forçar uma classe nesses exemplos aumentaria o ruído e daria uma falsa impressão de verdade-terreno.


In [ ]:
POSITIVE_PROTOTYPES = [
    "O cliente precisa de um novo sistema e quer receber uma proposta comercial.",
    "A empresa está avaliando migrar o ERP e pediu preço e prazo de implantação.",
    "Há interesse em contratar uma solução para eliminar processos manuais.",
    "O cliente relatou uma dor de integração e quer conhecer um produto TOTVS.",
    "A organização pretende substituir o fornecedor atual e solicitou uma cotação.",
    "Existe orçamento aprovado para implantar o módulo no próximo trimestre.",
]
NEGATIVE_PROTOTYPES = [
    "Os participantes apenas se cumprimentam e aguardam o início da reunião.",
    "A conversa trata somente de agenda, horário e presença dos convidados.",
    "O participante explica um procedimento sem pedir produto, proposta ou contratação.",
    "Não existe interesse em comprar ou trocar o sistema neste momento.",
    "A equipe encerra a reunião e combina o envio da ata.",
    "O trecho é neutro e não apresenta necessidade comercial do cliente.",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()


### Geração de representações semânticas

A função processa textos em lotes, aplica mean pooling com máscara e normalização L2. O resultado permite comparar chunks e protótipos por similaridade.


In [ ]:
def embed_texts(texts: list[str], batch_size: int = BATCH_SIZE) -> torch.Tensor:
    vectors: list[torch.Tensor] = []
    started = time.perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        with torch.inference_mode():
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                hidden = model(**encoded).last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1)
            pooled = (hidden.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled = F.normalize(pooled, p=2, dim=1)
        vectors.append(pooled.cpu())
    elapsed = time.perf_counter() - started
    return torch.cat(vectors), elapsed


### Preparação dos objetos desta etapa

A célula prepara `positive_centroid`, `negative_centroid`, `margins`, `bert_labels` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
prototype_vectors, prototype_seconds = embed_texts(POSITIVE_PROTOTYPES + NEGATIVE_PROTOTYPES)
positive_centroid = F.normalize(prototype_vectors[:len(POSITIVE_PROTOTYPES)].mean(dim=0), dim=0)
negative_centroid = F.normalize(prototype_vectors[len(POSITIVE_PROTOTYPES):].mean(dim=0), dim=0)
chunk_vectors, embedding_seconds = embed_texts([row["text"] for row in sample])
margins = (chunk_vectors @ positive_centroid - chunk_vectors @ negative_centroid).numpy()
low_threshold, high_threshold = np.quantile(margins, [0.30, 0.70]).tolist()
bert_labels = np.where(margins <= low_threshold, 0, np.where(margins >= high_threshold, 1, -1))

del model, chunk_vectors
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print({
    "bert_labels": dict(Counter(int(label) for label in bert_labels)),
    "margin_p30": round(low_threshold, 6),
    "margin_p70": round(high_threshold, 6),
    "embedding_seconds": round(embedding_seconds, 2),
})


## 6. Consolidação, fila humana e split de desenvolvimento

Uma concordância significa que os dois métodos deram o mesmo rótulo sem abstenção. Esse conjunto pode iniciar os experimentos, mas continua sendo `pseudo_label`. A fila humana é cega: contém texto e campos vazios de anotação, sem mostrar as previsões, para reduzir viés do revisor. Cada item vem de uma reunião diferente e todas essas reuniões ficam fora do desenvolvimento. O split `train/validation` é apenas de desenvolvimento; o teste final nascerá da auditoria humana.

Nesta seção, os sinais anteriores são reunidos e convertidos nos artefatos consumidos pela modelagem. A reserva para auditoria humana permanece isolada do desenvolvimento para evitar contaminação da avaliação futura.


In [ ]:
combined: list[dict] = []
for row, (rule_value, rule_score, rule_reasons), bert_value, margin in zip(sample, rule_results, bert_labels, margins):
    bert_value = int(bert_value)
    agreement = rule_value in {0, 1} and rule_value == bert_value
    combined.append({
        "meeting_id": str(row["meeting_id"]),
        "chunk_id": row["chunk_id"],
        "chunk_index": row["chunk_index"],
        "text": row["text"],
        "rule_label": rule_value,
        "rule_score": rule_score,
        "rule_reasons": rule_reasons,
        "bert_prototype_label": bert_value,
        "bert_margin": round(float(margin), 8),
        "pseudo_label": rule_value if agreement else None,
        "label_origin": "rule_and_bert_agreement" if agreement else "needs_review",
    })

pseudo_rows_all = [row for row in combined if row["pseudo_label"] is not None]
needs_review = [row for row in combined if row["pseudo_label"] is None]


### Reserva de exemplos por reunião

A seleção aceita no máximo um item por reunião para a fila humana. Isso amplia a diversidade da auditoria e mantém essas reuniões fora do desenvolvimento.


In [ ]:
def take_unique_meetings(
    rows: list[dict], predicate, amount: int, rng: random.Random, used_meetings: set[str]
) -> list[dict]:
    candidates = [row for row in rows if predicate(row)]
    rng.shuffle(candidates)
    selected = []
    for row in candidates:
        if row["meeting_id"] in used_meetings:
            continue
        selected.append(row)
        used_meetings.add(row["meeting_id"])
        if len(selected) == amount:
            break
    return selected


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
audit_rng = random.Random(SEED + 1)
per_group = AUDIT_SIZE // 3
reserved_meetings: set[str] = set()
audit_candidates = []
audit_candidates += take_unique_meetings(
    pseudo_rows_all, lambda row: row["pseudo_label"] == 1, per_group, audit_rng, reserved_meetings
)
audit_candidates += take_unique_meetings(
    pseudo_rows_all, lambda row: row["pseudo_label"] == 0, per_group, audit_rng, reserved_meetings
)
audit_candidates += take_unique_meetings(
    needs_review, lambda row: True, per_group, audit_rng, reserved_meetings
)
if len(audit_candidates) < AUDIT_SIZE:
    audit_candidates += take_unique_meetings(
        combined, lambda row: True, AUDIT_SIZE - len(audit_candidates), audit_rng, reserved_meetings
    )
audit_rng.shuffle(audit_candidates)
audit_rows = [{
    "meeting_id": row["meeting_id"],
    "chunk_id": row["chunk_id"],
    "text": row["text"],
    "human_label": None,
    "reviewer_notes": "",
} for row in audit_candidates[:AUDIT_SIZE]]
audit_meetings = {row["meeting_id"] for row in audit_rows}
assert len(audit_meetings) == len(audit_rows)

pseudo_rows = [row for row in pseudo_rows_all if row["meeting_id"] not in audit_meetings]
meeting_targets: dict[str, int] = defaultdict(int)


### Divisão agrupada de desenvolvimento

As reuniões são separadas antes de atribuir cada chunk ao treino ou à validação. As asserções seguintes confirmam que não existe sobreposição entre as partições.


In [ ]:
for row in pseudo_rows:
    meeting_targets[row["meeting_id"]] = max(meeting_targets[row["meeting_id"]], row["pseudo_label"])
meeting_ids = sorted(meeting_targets)
strata = [meeting_targets[meeting_id] for meeting_id in meeting_ids]
stratify = strata if len(set(strata)) == 2 and min(Counter(strata).values()) >= 2 else None
train_meetings, validation_meetings = train_test_split(
    meeting_ids, test_size=0.20, random_state=SEED, stratify=stratify
)
train_meetings, validation_meetings = set(train_meetings), set(validation_meetings)
assert train_meetings.isdisjoint(validation_meetings)
assert (train_meetings | validation_meetings).isdisjoint(audit_meetings)
for row in pseudo_rows:
    row["weak_split"] = "train" if row["meeting_id"] in train_meetings else "validation"

PSEUDO_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
with PSEUDO_PATH.open("w", encoding="utf-8") as file:
    for row in pseudo_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")
with AUDIT_PATH.open("w", encoding="utf-8") as file:
    for row in audit_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

co_labeled = [row for row in combined if row["rule_label"] in {0, 1} and row["bert_prototype_label"] in {0, 1}]


### Consolidação do relatório da etapa

Configurações, contagens, métricas e alertas são reunidos em um resumo auditável. O relatório registra resultados agregados sem acrescentar novas transformações aos dados.


In [ ]:
summary = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "device": str(DEVICE),
    "device_name": device_name,
    "torch_version": torch.__version__,
    "cuda_build": torch.version.cuda,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "sample_chunks": len(sample),
    "sample_meetings": len({row["meeting_id"] for row in sample}),
    "rule_label_counts": dict(Counter(str(row["rule_label"]) for row in combined)),
    "bert_label_counts": dict(Counter(str(row["bert_prototype_label"]) for row in combined)),
    "bert_margin_p30": low_threshold,
    "bert_margin_p70": high_threshold,
    "pseudo_label_counts_before_audit_reserve": dict(Counter(str(row["pseudo_label"]) for row in pseudo_rows_all)),
    "pseudo_label_count_before_audit_reserve": len(pseudo_rows_all),
    "pseudo_label_coverage": len(pseudo_rows_all) / len(combined),
    "development_pseudo_label_counts": dict(Counter(str(row["pseudo_label"]) for row in pseudo_rows)),
    "development_pseudo_label_count": len(pseudo_rows),
    "agreement_among_co_labeled": (len(pseudo_rows_all) / len(co_labeled)) if co_labeled else None,
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "split_leakage_meetings": len(train_meetings & validation_meetings),
    "audit_queue_size": len(audit_rows),
    "audit_meetings": len(audit_meetings),
    "audit_development_leakage_meetings": len(audit_meetings & (train_meetings | validation_meetings)),
    "prototype_embedding_seconds": prototype_seconds,
    "chunk_embedding_seconds": embedding_seconds,
    "metric_warning": "Agreement and coverage are not precision. Final metrics require human ground truth.",
}


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [ ]:
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print({
    "pseudo_labels_before_audit_reserve": len(pseudo_rows_all),
    "development_pseudo_labels": len(pseudo_rows),
    "development_class_balance": summary["development_pseudo_label_counts"],
    "coverage": round(summary["pseudo_label_coverage"], 4),
    "agreement_among_co_labeled": round(summary["agreement_among_co_labeled"], 4) if summary["agreement_among_co_labeled"] is not None else None,
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "audit_queue": len(audit_rows),
    "split_leakage": summary["split_leakage_meetings"],
    "audit_development_leakage": summary["audit_development_leakage_meetings"],
})


## 7. Como interpretar e continuar

- `pseudo_labels_opportunity.jsonl` serve para prototipar TF-IDF + Logistic Regression e o fine-tuning do BERTimbau. O campo `label_origin` impede confusão com rótulos humanos.
- `annotation_queue_opportunity.jsonl` deve ser preenchido por uma pessoa com `human_label = 0` ou `1`. Casos ambíguos podem receber `null` e uma observação.
- A primeira avaliação honesta é comparar os pseudo-rótulos com essa auditoria humana e calcular precisão por origem/classe.
- Depois da auditoria, congelaremos reuniões humanas exclusivas para teste e treinaremos os dois modelos exigidos pela Sprint 3 no mesmo split.
- Similaridade de BERTimbau base não substitui fine-tuning; ela apenas reduz o custo de iniciar a anotação.

A leitura dos resultados deve considerar tanto o padrão observado quanto as limitações da referência. As conclusões desta seção orientam a próxima etapa, mas não substituem a validação humana prevista no projeto.


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 6 de 11 — 06 baseline tfidf logreg

Esta seção reproduz a etapa `06_baseline_tfidf_logreg.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 06 — Baseline TF-IDF + Regressão Logística

Este notebook treina o primeiro modelo supervisionado da Sprint 3 para classificar chunks em `oportunidade` (1) e `nao_oportunidade` (0). O baseline combina TF-IDF de unigramas e bigramas com Regressão Logística.

> **Limite metodológico:** os alvos atuais são **pseudo-rótulos** produzidos pela concordância entre regras e BERTimbau. Portanto, as métricas deste notebook medem reprodução desses pseudo-rótulos, não desempenho real de negócio. A avaliação final exigirá os rótulos humanos da fila de auditoria.

**Objetivo e conexão com o projeto.** O baseline transforma os textos em atributos TF-IDF e ajusta uma Regressão Logística no split produzido anteriormente. Além de oferecer uma referência rápida e interpretável, ele permite verificar quanto do sinal dos pseudo-rótulos pode ser capturado por padrões lexicais antes de avaliar o transformer.


## 1. Decisões

- Usamos o split `train/validation` criado no notebook 05 e validamos novamente a ausência de reuniões compartilhadas.
- O TF-IDF é ajustado exclusivamente no treino para evitar vazamento de vocabulário.
- Unigramas capturam termos comerciais; bigramas capturam expressões como `proposta comercial` e `sem interesse`.
- `min_df=2` reduz termos acidentais e `max_df=0.98` remove termos quase universais.
- `class_weight=balanced` reduz o efeito do desbalanceamento inicial.
- O limiar permanece em 0,5; não usamos a validação para otimizá-lo nesta primeira referência.
- A métrica principal é o recall da classe oportunidade, pois perder uma oportunidade relevante tende a ser mais custoso do que encaminhar um falso positivo para revisão.
- Nenhuma transcrição é exibida ou salva nos relatórios versionáveis.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import tempfile
import time
from collections import Counter
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "wedjat-matplotlib"))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

SEED = 42
THRESHOLD = 0.5


### Função auxiliar: `find_project_root`

Esta célula isola a responsabilidade implementada por `find_project_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada. No Colab, entre na pasta Wedjat.")


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `DATA_PATH`, `METRICS_PATH`, `FIGURE_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT = find_project_root()
DATA_PATH = ROOT / "data/processed/pseudo_labels_opportunity.jsonl"
METRICS_PATH = ROOT / "reports/metrics/baseline_tfidf_logreg_metrics.json"
FIGURE_PATH = ROOT / "reports/figures/baseline_tfidf_logreg_confusion_matrix.png"
PREDICTIONS_PATH = ROOT / "data/processed/baseline_tfidf_logreg_validation_predictions.jsonl"
ERRORS_PATH = ROOT / "data/processed/baseline_tfidf_logreg_errors.jsonl"
MODEL_PATH = ROOT / "data/processed/baseline_tfidf_logreg.joblib"

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
})


## 3. Leitura e validação do split

Além de conferir esquema e classes, a célula interrompe a execução caso uma reunião apareça simultaneamente no treino e na validação.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
rows = read_jsonl(DATA_PATH)
required_fields = {"meeting_id", "chunk_id", "text", "pseudo_label", "weak_split"}
assert rows, "Arquivo de pseudo-rótulos vazio. Execute o notebook 05."
assert all(required_fields <= row.keys() for row in rows), "Esquema de entrada inválido."
assert {row["pseudo_label"] for row in rows} == {0, 1}, "As duas classes precisam estar presentes."
assert {row["weak_split"] for row in rows} == {"train", "validation"}

train_rows = [row for row in rows if row["weak_split"] == "train"]
validation_rows = [row for row in rows if row["weak_split"] == "validation"]
train_meetings = {row["meeting_id"] for row in train_rows}
validation_meetings = {row["meeting_id"] for row in validation_rows}
meeting_overlap = train_meetings & validation_meetings
assert not meeting_overlap, f"Vazamento detectado em {len(meeting_overlap)} reuniões."

X_train_text = [row["text"] for row in train_rows]
y_train = np.asarray([row["pseudo_label"] for row in train_rows], dtype=np.int64)
X_validation_text = [row["text"] for row in validation_rows]
y_validation = np.asarray([row["pseudo_label"] for row in validation_rows], dtype=np.int64)

print({
    "train_chunks": len(train_rows),
    "validation_chunks": len(validation_rows),
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "train_labels": dict(Counter(y_train.tolist())),
    "validation_labels": dict(Counter(y_validation.tolist())),
    "meeting_overlap": len(meeting_overlap),
})


## 4. Treinamento

O `TfidfVectorizer` e a Regressão Logística são armazenados juntos apenas em `data/processed/`, pasta ignorada pelo Git. Isso preserva a reprodutibilidade local sem versionar um vocabulário derivado das transcrições.

O ajuste usa somente a partição de treino e conserva a validação para seleção e comparação. Hiperparâmetros, balanceamento e critério de checkpoint são registrados para tornar o experimento reproduzível.


In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=50_000,
    sublinear_tf=True,
    dtype=np.float32,
)
classifier = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2_000,
    solver="liblinear",
    random_state=SEED,
)

fit_started = time.perf_counter()
X_train = vectorizer.fit_transform(X_train_text)
classifier.fit(X_train, y_train)
fit_seconds = time.perf_counter() - fit_started

transform_started = time.perf_counter()
X_validation = vectorizer.transform(X_validation_text)
validation_probabilities = classifier.predict_proba(X_validation)[:, 1]
inference_seconds = time.perf_counter() - transform_started
validation_predictions = (validation_probabilities >= THRESHOLD).astype(np.int64)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
joblib.dump({"vectorizer": vectorizer, "classifier": classifier}, MODEL_PATH)

print({
    "features": len(vectorizer.vocabulary_),
    "fit_seconds": round(fit_seconds, 4),
    "validation_inference_seconds": round(inference_seconds, 4),
    "iterations": int(classifier.n_iter_[0]),
})


## 5. Métricas no conjunto de validação

Relatamos accuracy, precision, recall e F1 para a classe positiva, além das médias macro e weighted. A matriz segue a ordem `[[TN, FP], [FN, TP]]`.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
matrix = confusion_matrix(y_validation, validation_predictions, labels=[0, 1])
report = classification_report(
    y_validation,
    validation_predictions,
    labels=[0, 1],
    target_names=["nao_oportunidade", "oportunidade"],
    output_dict=True,
    zero_division=0,
)
metrics = {
    "accuracy": float(accuracy_score(y_validation, validation_predictions)),
    "precision_oportunidade": float(precision_score(y_validation, validation_predictions, pos_label=1, zero_division=0)),
    "recall_oportunidade": float(recall_score(y_validation, validation_predictions, pos_label=1, zero_division=0)),
    "f1_oportunidade": float(f1_score(y_validation, validation_predictions, pos_label=1, zero_division=0)),
    "precision_macro": float(precision_score(y_validation, validation_predictions, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_validation, validation_predictions, average="macro", zero_division=0)),
    "f1_macro": float(f1_score(y_validation, validation_predictions, average="macro", zero_division=0)),
    "precision_weighted": float(precision_score(y_validation, validation_predictions, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(y_validation, validation_predictions, average="weighted", zero_division=0)),
    "f1_weighted": float(f1_score(y_validation, validation_predictions, average="weighted", zero_division=0)),
}
tn, fp, fn, tp = (int(value) for value in matrix.ravel())

print({
    "accuracy": round(metrics["accuracy"], 4),
    "precision_oportunidade": round(metrics["precision_oportunidade"], 4),
    "recall_oportunidade": round(metrics["recall_oportunidade"], 4),
    "f1_oportunidade": round(metrics["f1_oportunidade"], 4),
    "f1_macro": round(metrics["f1_macro"], 4),
    "confusion_matrix": matrix.tolist(),
})

### Visualização da matriz de confusão

A matriz apresenta acertos e erros por classe. Ela complementa as métricas agregadas ao mostrar diretamente quais tipos de erro ocorreram.


In [ ]:
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
display = ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=["Não oportunidade", "Oportunidade"]
)
display.plot(cmap="Blues", values_format="d", colorbar=False)
display.ax_.set_xlabel("Rótulo predito")
display.ax_.set_ylabel("Rótulo verdadeiro (pseudo-rótulo)")
plt.title("TF-IDF + Regressão Logística — validação com pseudo-rótulos")
plt.tight_layout()
plt.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()

## 6. Artefatos e análise de erros

O relatório versionável contém somente números agregados. As previsões e o manifesto de erros ficam em `data/processed/`, sem o texto, e permitem localizar os chunks para revisão controlada.

Os artefatos persistem o que será consumido pelas próximas etapas e separam resultados agregados de dados sensíveis. IDs, configurações e métricas permitem auditoria sem copiar o texto das reuniões.


In [ ]:
prediction_rows = []
for row, expected, predicted, probability in zip(
    validation_rows, y_validation, validation_predictions, validation_probabilities
):
    prediction_rows.append({
        "meeting_id": row["meeting_id"],
        "chunk_id": row["chunk_id"],
        "pseudo_label": int(expected),
        "prediction": int(predicted),
        "probability_oportunidade": round(float(probability), 8),
        "correct_against_pseudo_label": bool(expected == predicted),
    })
error_rows = [row for row in prediction_rows if not row["correct_against_pseudo_label"]]

for path, output_rows in [(PREDICTIONS_PATH, prediction_rows), (ERRORS_PATH, error_rows)]:
    with path.open("w", encoding="utf-8") as file:
        for row in output_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")


### Consolidação do relatório da etapa

Configurações, contagens, métricas e alertas são reunidos em um resumo auditável. O relatório registra resultados agregados sem acrescentar novas transformações aos dados.


In [ ]:
summary = {
    "schema_version": "1.0",
    "model": "TF-IDF + LogisticRegression",
    "target": "oportunidade_comercial_binaria_no_chunk",
    "evaluation_reference": "pseudo_labels_rule_and_bert_agreement",
    "warning": "Estas métricas não estimam desempenho real; falta avaliação contra rótulos humanos.",
    "seed": SEED,
    "threshold": THRESHOLD,
    "train_chunks": len(train_rows),
    "validation_chunks": len(validation_rows),
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "meeting_overlap": len(meeting_overlap),
    "train_label_counts": dict(Counter(str(value) for value in y_train.tolist())),
    "validation_label_counts": dict(Counter(str(value) for value in y_validation.tolist())),
    "vectorizer": {
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_df": 0.98,
        "max_features": 50000,
        "actual_features": len(vectorizer.vocabulary_),
        "sublinear_tf": True,
    },
    "classifier": {
        "C": 1.0,
        "class_weight": "balanced",
        "solver": "liblinear",
        "iterations": int(classifier.n_iter_[0]),
    },
    "metrics": metrics,
    "classification_report": report,
    "confusion_matrix_order": [["TN", "FP"], ["FN", "TP"]],
    "confusion_matrix": matrix.tolist(),
    "error_counts": {"false_positive": fp, "false_negative": fn, "total": len(error_rows)},
    "primary_metric": {
        "name": "recall_oportunidade",
        "value": metrics["recall_oportunidade"],
        "justification": "Perder uma oportunidade relevante tende a ser mais custoso do que revisar um falso positivo.",
    },
    "fit_seconds": fit_seconds,
    "validation_inference_seconds": inference_seconds,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
}


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print({
    "saved_metrics": str(METRICS_PATH.relative_to(ROOT)),
    "saved_figure": str(FIGURE_PATH.relative_to(ROOT)),
    "validation_predictions": len(prediction_rows),
    "errors_without_text": len(error_rows),
})


## 7. Interpretação e próximo passo

Este resultado estabelece uma referência barata, rápida e interpretável. Como parte dos pseudo-rótulos veio de sinais lexicais, o TF-IDF pode reproduzir as regras e apresentar uma estimativa otimista. O notebook 07 deverá treinar o BERTimbau com o mesmo split e o notebook 08 fará a comparação. Nenhum dos dois substituirá a avaliação final no conjunto humano reservado.

A leitura dos resultados deve considerar tanto o padrão observado quanto as limitações da referência. As conclusões desta seção orientam a próxima etapa, mas não substituem a validação humana prevista no projeto.


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 7 de 11 — 07 bertimbau finetuning

Esta seção reproduz a etapa `07_bertimbau_finetuning.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 07 — Fine-tuning do BERTimbau

Este notebook ajusta o checkpoint `neuralmind/bert-base-portuguese-cased` para a classificação binária de oportunidade comercial no nível do chunk. O treino usa exatamente as mesmas reuniões e os mesmos pseudo-rótulos do baseline TF-IDF + Regressão Logística.

> **Limite metodológico:** a validação atual usa pseudo-rótulos. As métricas permitem comparar os dois algoritmos sob a mesma referência, mas não medem ainda a qualidade real contra decisões humanas.

**Objetivo e conexão com o projeto.** Esta etapa ajusta o BERTimbau para o mesmo alvo, dados e partições usados pelo baseline, tornando a comparação posterior tecnicamente coerente. O treinamento preserva o limite de 512 tokens, aplica pesos de classe e seleciona o checkpoint pela validação. As métricas continuam sendo uma pseudo-validação, não evidência de desempenho em produção.


## 1. Decisões de treinamento

- Checkpoint: `neuralmind/bert-base-portuguese-cased`.
- Comprimento máximo: 512 tokens, preservando os 510 tokens de conteúdo definidos no chunking mais os tokens especiais do BERT.
- Batch físico 4 e acumulação de 2 passos produzem batch efetivo 8.
- Precisão mista FP16 é ativada apenas quando CUDA está disponível.
- Pesos de classe compensam o desbalanceamento dos pseudo-rótulos de treino.
- AdamW, learning rate `2e-5`, weight decay `0.01`, warmup linear de 10% e no máximo 3 épocas.
- O melhor checkpoint é escolhido pelo F1 da classe oportunidade; o recall dessa classe permanece a métrica principal de negócio.
- O split por reunião é validado novamente antes do treino.
- Textos das transcrições não aparecem nas saídas do notebook nem nos relatórios versionáveis.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
import tempfile
import time
import warnings
from collections import Counter
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "wedjat-matplotlib"))
warnings.filterwarnings("ignore", message="IProgress not found.*")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


### Preparação do ambiente e das dependências

As bibliotecas são reunidas antes do processamento e organizam os recursos de dados, modelagem, avaliação e persistência usados nesta etapa. Parâmetros e caminhos permanecem explícitos para facilitar a reprodução local e no Colab.


In [ ]:
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 512
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


### Função auxiliar: `find_project_root`

Esta célula isola a responsabilidade implementada por `find_project_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada. No Colab, entre na pasta Wedjat.")


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `DATA_PATH`, `CHECKPOINT_DIR`, `PREDICTIONS_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT = find_project_root()
DATA_PATH = ROOT / "data/processed/pseudo_labels_opportunity.jsonl"
CHECKPOINT_DIR = ROOT / "data/processed/bertimbau_opportunity_best"
PREDICTIONS_PATH = ROOT / "data/processed/bertimbau_validation_predictions.jsonl"
ERRORS_PATH = ROOT / "data/processed/bertimbau_validation_errors.jsonl"
METRICS_PATH = ROOT / "reports/metrics/bertimbau_finetuning_metrics.json"
FIGURE_PATH = ROOT / "reports/figures/bertimbau_confusion_matrix.png"

device_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print({
    "device": str(DEVICE),
    "device_name": device_name,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
})


## 3. Dados e proteção contra vazamento

O notebook usa o arquivo produzido pelo notebook 05. Nenhum novo sorteio é feito: essa é a condição necessária para comparar BERTimbau e baseline no mesmo conjunto.

Antes de transformar ou modelar, conferimos os dados usados nesta etapa e as condições que garantem comparabilidade. As verificações protegem a sequência do notebook contra arquivos incompletos e partições incompatíveis.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
rows = read_jsonl(DATA_PATH)
required_fields = {"meeting_id", "chunk_id", "text", "pseudo_label", "weak_split"}
assert rows and all(required_fields <= row.keys() for row in rows)
assert {row["pseudo_label"] for row in rows} == {0, 1}

train_rows = [row for row in rows if row["weak_split"] == "train"]
validation_rows = [row for row in rows if row["weak_split"] == "validation"]
train_meetings = {row["meeting_id"] for row in train_rows}
validation_meetings = {row["meeting_id"] for row in validation_rows}
meeting_overlap = train_meetings & validation_meetings
assert not meeting_overlap, f"Vazamento entre {len(meeting_overlap)} reuniões."

y_train = np.asarray([row["pseudo_label"] for row in train_rows], dtype=np.int64)
y_validation = np.asarray([row["pseudo_label"] for row in validation_rows], dtype=np.int64)

print({
    "train_chunks": len(train_rows),
    "validation_chunks": len(validation_rows),
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "train_labels": dict(Counter(y_train.tolist())),
    "validation_labels": dict(Counter(y_validation.tolist())),
    "meeting_overlap": len(meeting_overlap),
})


## 4. Tokenização e DataLoaders

A tokenização usa o limite completo de 512 tokens e padding dinâmico por batch. O padding em múltiplos de 8 melhora o aproveitamento dos Tensor Cores quando CUDA está ativa.

A tokenização traduz o texto para a representação aceita pelo modelo. Conferir tokens especiais, limites e máscaras é necessário para que cada entrada respeite a arquitetura e para que o chunking não dependa de uma estimativa por caracteres.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


### Dataset tokenizado para o BERTimbau

A classe encapsula a tokenização dos textos e seus rótulos no formato esperado pelos DataLoaders, mantendo a preparação de entradas separada do laço de treinamento.


In [ ]:
class EncodedChunkDataset(Dataset):
    def __init__(self, source_rows: list[dict]):
        self.encodings = tokenizer(
            [row["text"] for row in source_rows],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        self.labels = [int(row["pseudo_label"]) for row in source_rows]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict:
        item = {key: values[index] for key, values in self.encodings.items()}
        item["labels"] = self.labels[index]
        return item


### Preparação dos objetos desta etapa

A célula prepara `data_collator`, `loader_generator`, `train_loader`, `validation_loader` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if DEVICE.type == "cuda" else None
)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    EncodedChunkDataset(train_rows),
    batch_size=PHYSICAL_BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    EncodedChunkDataset(validation_rows),
    batch_size=PHYSICAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)

raw_lengths = [len(tokenizer.tokenize(row["text"])) + 2 for row in rows]
effective_lengths = [min(length, MAX_LENGTH) for length in raw_lengths]


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
print({
    "max_length": MAX_LENGTH,
    "median_effective_length": float(np.median(effective_lengths)),
    "max_raw_length": max(raw_lengths),
    "fraction_truncated": round(sum(length > MAX_LENGTH for length in raw_lengths) / len(raw_lengths), 4),
    "train_batches_per_epoch": len(train_loader),
})


## 5. Modelo, pesos de classe e treinamento

A função de perda recebe pesos inversamente proporcionais à frequência de cada classe no treino. Gradient checkpointing e FP16 reduzem o uso de memória da GPU.

O ajuste usa somente a partição de treino e conserva a validação para seleção e comparação. Hiperparâmetros, balanceamento e critério de checkpoint são registrados para tornar o experimento reproduzível.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "nao_oportunidade", 1: "oportunidade"},
    label2id={"nao_oportunidade": 0, "oportunidade": 1},
).to(DEVICE)
model.gradient_checkpointing_enable()
model.config.use_cache = False

class_counts = np.bincount(y_train, minlength=2)
class_weights = torch.tensor(
    len(y_train) / (2.0 * class_counts), dtype=torch.float32, device=DEVICE
)

no_decay = ("bias", "LayerNorm.weight")
optimizer_groups = [
    {
        "params": [parameter for name, parameter in model.named_parameters() if not any(term in name for term in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [parameter for name, parameter in model.named_parameters() if any(term in name for term in no_decay)],
        "weight_decay": 0.0,
    },
]
optimizer = AdamW(optimizer_groups, lr=LEARNING_RATE)
updates_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_update_steps = updates_per_epoch * EPOCHS
warmup_steps = max(1, round(total_update_steps * 0.10))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)


### Preparação dos objetos desta etapa

A célula prepara `scaler` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print({
    "parameters": sum(parameter.numel() for parameter in model.parameters()),
    "class_weights": [round(value, 4) for value in class_weights.detach().cpu().tolist()],
    "updates_per_epoch": updates_per_epoch,
    "total_update_steps": total_update_steps,
    "warmup_steps": warmup_steps,
})


### Função de avaliação

A função concentra o cálculo de perdas, probabilidades, classes e métricas. Isolá-la garante que as mesmas regras sejam usadas durante a seleção e na avaliação final do checkpoint.


In [ ]:
def evaluate(current_model, loader: DataLoader) -> dict:
    current_model.eval()
    losses, labels_all, probabilities_all = [], [], []
    with torch.inference_mode():
        for batch in loader:
            batch = batch.to(DEVICE)
            labels = batch.pop("labels")
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = current_model(**batch).logits
                loss = F.cross_entropy(logits, labels, weight=class_weights)
            probabilities = torch.softmax(logits.float(), dim=-1)[:, 1]
            losses.extend([float(loss.item())] * labels.shape[0])
            labels_all.extend(labels.cpu().tolist())
            probabilities_all.extend(probabilities.cpu().tolist())
    labels_array = np.asarray(labels_all, dtype=np.int64)
    probabilities_array = np.asarray(probabilities_all, dtype=np.float64)
    predictions_array = (probabilities_array >= THRESHOLD).astype(np.int64)
    return {
        "loss": float(np.mean(losses)),
        "labels": labels_array,
        "probabilities": probabilities_array,
        "predictions": predictions_array,
        "accuracy": float(accuracy_score(labels_array, predictions_array)),
        "precision_oportunidade": float(precision_score(labels_array, predictions_array, zero_division=0)),
        "recall_oportunidade": float(recall_score(labels_array, predictions_array, zero_division=0)),
        "f1_oportunidade": float(f1_score(labels_array, predictions_array, zero_division=0)),
        "f1_macro": float(f1_score(labels_array, predictions_array, average="macro", zero_division=0)),
    }


### Preparação dos objetos desta etapa

A célula prepara `history`, `best_f1`, `best_epoch`, `training_started` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
history = []
best_f1 = -1.0
best_epoch = 0
training_started = time.perf_counter()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()


### Laço de treinamento por épocas

O modelo é ajustado no conjunto de treino e avaliado ao final de cada época. O melhor estado é persistido segundo o critério de validação definido no experimento.


In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    epoch_loss = 0.0
    examples_seen = 0

    for step, batch in enumerate(train_loader, start=1):
        batch = batch.to(DEVICE)
        labels = batch.pop("labels")
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
            loss = F.cross_entropy(logits, labels, weight=class_weights)
            scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
        scaler.scale(scaled_loss).backward()
        epoch_loss += float(loss.item()) * labels.shape[0]
        examples_seen += labels.shape[0]

        should_update = step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader)
        if should_update:
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    validation_result = evaluate(model, validation_loader)
    epoch_result = {
        "epoch": epoch,
        "train_loss": epoch_loss / examples_seen,
        "validation_loss": validation_result["loss"],
        "accuracy": validation_result["accuracy"],
        "precision_oportunidade": validation_result["precision_oportunidade"],
        "recall_oportunidade": validation_result["recall_oportunidade"],
        "f1_oportunidade": validation_result["f1_oportunidade"],
        "f1_macro": validation_result["f1_macro"],
    }
    history.append(epoch_result)
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in epoch_result.items()})

    if validation_result["f1_oportunidade"] > best_f1:
        best_f1 = validation_result["f1_oportunidade"]
        best_epoch = epoch
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(CHECKPOINT_DIR)
        tokenizer.save_pretrained(CHECKPOINT_DIR)


### Preparação dos objetos desta etapa

A célula prepara `training_seconds`, `peak_gpu_memory_mb` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
training_seconds = time.perf_counter() - training_started
peak_gpu_memory_mb = (
    torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else 0.0
)
print({
    "best_epoch": best_epoch,
    "training_seconds": round(training_seconds, 2),
    "peak_gpu_memory_mb": round(peak_gpu_memory_mb, 2),
})


## 6. Avaliação do melhor checkpoint

Recarregamos o melhor checkpoint antes de calcular a matriz e o relatório completo. A matriz segue `[[TN, FP], [FN, TP]]`.

A avaliação transforma as previsões em medidas comparáveis e mantém a unidade de análise explícita. Como os rótulos atuais são automáticos, os números descrevem o experimento de desenvolvimento e não desempenho comprovado em produção.


In [ ]:
del model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
best_model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR).to(DEVICE)
final_result = evaluate(best_model, validation_loader)
final_labels = final_result.pop("labels")
final_probabilities = final_result.pop("probabilities")
final_predictions = final_result.pop("predictions")
matrix = confusion_matrix(final_labels, final_predictions, labels=[0, 1])
report = classification_report(
    final_labels,
    final_predictions,
    labels=[0, 1],
    target_names=["nao_oportunidade", "oportunidade"],
    output_dict=True,
    zero_division=0,
)
full_metrics = {
    "loss": final_result["loss"],
    "accuracy": final_result["accuracy"],
    "precision_oportunidade": final_result["precision_oportunidade"],
    "recall_oportunidade": final_result["recall_oportunidade"],
    "f1_oportunidade": final_result["f1_oportunidade"],
    "precision_macro": float(precision_score(final_labels, final_predictions, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(final_labels, final_predictions, average="macro", zero_division=0)),
    "f1_macro": final_result["f1_macro"],
    "precision_weighted": float(precision_score(final_labels, final_predictions, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(final_labels, final_predictions, average="weighted", zero_division=0)),
    "f1_weighted": float(f1_score(final_labels, final_predictions, average="weighted", zero_division=0)),
}


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
tn, fp, fn, tp = (int(value) for value in matrix.ravel())

print({
    "best_epoch": best_epoch,
    "accuracy": round(full_metrics["accuracy"], 4),
    "precision_oportunidade": round(full_metrics["precision_oportunidade"], 4),
    "recall_oportunidade": round(full_metrics["recall_oportunidade"], 4),
    "f1_oportunidade": round(full_metrics["f1_oportunidade"], 4),
    "f1_macro": round(full_metrics["f1_macro"], 4),
    "confusion_matrix": matrix.tolist(),
})


### Visualização da matriz de confusão

A matriz apresenta acertos e erros por classe. Ela complementa as métricas agregadas ao mostrar diretamente quais tipos de erro ocorreram.


In [ ]:
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
display = ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=["Não oportunidade", "Oportunidade"]
)
display.plot(cmap="Purples", values_format="d", colorbar=False)
display.ax_.set_xlabel("Rótulo predito")
display.ax_.set_ylabel("Rótulo verdadeiro (pseudo-rótulo)")
plt.title("BERTimbau — validação com pseudo-rótulos")
plt.tight_layout()
plt.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()

## 7. Artefatos e relatório

As previsões e os erros ficam na pasta processada sem copiar textos. O relatório versionável contém configuração, histórico, métricas e tempos agregados.

Os artefatos persistem o que será consumido pelas próximas etapas e separam resultados agregados de dados sensíveis. IDs, configurações e métricas permitem auditoria sem copiar o texto das reuniões.


In [ ]:
prediction_rows = []
for row, expected, predicted, probability in zip(
    validation_rows, final_labels, final_predictions, final_probabilities
):
    prediction_rows.append({
        "meeting_id": row["meeting_id"],
        "chunk_id": row["chunk_id"],
        "pseudo_label": int(expected),
        "prediction": int(predicted),
        "probability_oportunidade": round(float(probability), 8),
        "correct_against_pseudo_label": bool(expected == predicted),
    })
error_rows = [row for row in prediction_rows if not row["correct_against_pseudo_label"]]
for path, output_rows in [(PREDICTIONS_PATH, prediction_rows), (ERRORS_PATH, error_rows)]:
    with path.open("w", encoding="utf-8") as file:
        for row in output_rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")


### Consolidação do relatório da etapa

Configurações, contagens, métricas e alertas são reunidos em um resumo auditável. O relatório registra resultados agregados sem acrescentar novas transformações aos dados.


In [ ]:
summary = {
    "schema_version": "1.0",
    "model": "BERTimbau fine-tuned for sequence classification",
    "checkpoint": MODEL_NAME,
    "target": "oportunidade_comercial_binaria_no_chunk",
    "evaluation_reference": "pseudo_labels_rule_and_bert_agreement",
    "warning": "Estas métricas não estimam desempenho real; falta avaliação contra rótulos humanos.",
    "seed": SEED,
    "threshold": THRESHOLD,
    "train_chunks": len(train_rows),
    "validation_chunks": len(validation_rows),
    "train_meetings": len(train_meetings),
    "validation_meetings": len(validation_meetings),
    "meeting_overlap": len(meeting_overlap),
    "train_label_counts": dict(Counter(str(value) for value in y_train.tolist())),
    "validation_label_counts": dict(Counter(str(value) for value in y_validation.tolist())),
    "training_config": {
        "max_length": MAX_LENGTH,
        "epochs_requested": EPOCHS,
        "best_epoch": best_epoch,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "physical_batch_size": PHYSICAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "warmup_steps": warmup_steps,
        "class_weights": class_weights.detach().cpu().tolist(),
        "fp16": USE_AMP,
        "gradient_checkpointing": True,
    },
    "history": history,
    "metrics": full_metrics,
    "classification_report": report,
    "confusion_matrix_order": [["TN", "FP"], ["FN", "TP"]],
    "confusion_matrix": matrix.tolist(),
    "error_counts": {"false_positive": fp, "false_negative": fn, "total": len(error_rows)},
    "primary_metric": {
        "name": "recall_oportunidade",
        "value": full_metrics["recall_oportunidade"],
        "justification": "Perder uma oportunidade relevante tende a ser mais custoso do que revisar um falso positivo.",
    },
    "hardware": {
        "device": str(DEVICE),
        "device_name": device_name,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
    },
    "training_seconds": training_seconds,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print({
    "saved_checkpoint": str(CHECKPOINT_DIR.relative_to(ROOT)),
    "saved_metrics": str(METRICS_PATH.relative_to(ROOT)),
    "saved_figure": str(FIGURE_PATH.relative_to(ROOT)),
    "validation_predictions": len(prediction_rows),
    "errors_without_text": len(error_rows),
})


## 8. Próximo passo

O notebook 08 deverá ler os relatórios do baseline e do BERTimbau, conferir que ambos usaram o mesmo split e construir a comparação lado a lado. A conclusão final deve destacar que a pseudo-validação favorece sinais usados na criação dos próprios rótulos; somente a auditoria humana permitirá afirmar precisão real.

Os itens a seguir separam o que já foi demonstrado do que ainda depende de dados humanos, calibração ou evolução técnica. Assim, o notebook termina sem transformar resultados experimentais em garantias de produção.


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 8 de 11 — 08 model comparison

Esta seção reproduz a etapa `08_model_comparison.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 08 — Comparação dos modelos

Este notebook compara TF-IDF + Regressão Logística e BERTimbau no mesmo conjunto de validação. A comparação inclui as métricas exigidas pela Sprint 3, matrizes de confusão, análise pareada dos erros, teste de McNemar e intervalos por bootstrap agrupado por reunião.

> **Limite metodológico:** a referência ainda são pseudo-rótulos. O resultado serve para comparar os algoritmos dentro deste experimento, mas a decisão final depende do conjunto humano reservado.

**Objetivo e conexão com o projeto.** Os dois classificadores são comparados caso a caso no mesmo conjunto de validação, com métricas de classe, análise de discordâncias e incerteza agrupada por reunião. A decisão resultante é provisória: ela combina qualidade preditiva, custo dos erros e custo computacional, mas permanece condicionada ao futuro conjunto de teste humano.


## 1. Critério de decisão

O **recall da classe oportunidade** é a métrica principal: um falso negativo pode esconder uma oportunidade comercial que não chegará ao time responsável. Precision continua importante para controlar o volume de revisões, enquanto F1 resume esse equilíbrio. Accuracy não deve ser usada isoladamente porque pode mascarar desempenho desigual entre classes.

A análise é pareada por `chunk_id`, e o bootstrap reamostra reuniões inteiras. Assim, respeitamos a dependência existente entre chunks da mesma reunião.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json
import os
import tempfile
from collections import defaultdict
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "wedjat-matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import scipy
import sklearn
from scipy.stats import binomtest
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42
BOOTSTRAP_REPETITIONS = 2_000


### Função auxiliar: `find_project_root`

Esta célula isola a responsabilidade implementada por `find_project_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada. No Colab, entre na pasta Wedjat.")


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `BASELINE_METRICS_PATH`, `BERT_METRICS_PATH`, `BASELINE_PREDICTIONS_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT = find_project_root()
BASELINE_METRICS_PATH = ROOT / "reports/metrics/baseline_tfidf_logreg_metrics.json"
BERT_METRICS_PATH = ROOT / "reports/metrics/bertimbau_finetuning_metrics.json"
BASELINE_PREDICTIONS_PATH = ROOT / "data/processed/baseline_tfidf_logreg_validation_predictions.jsonl"
BERT_PREDICTIONS_PATH = ROOT / "data/processed/bertimbau_validation_predictions.jsonl"
BASELINE_MODEL_PATH = ROOT / "data/processed/baseline_tfidf_logreg.joblib"
BERT_MODEL_PATH = ROOT / "data/processed/bertimbau_opportunity_best/model.safetensors"
REPORT_PATH = ROOT / "reports/metrics/model_comparison.json"
METRICS_FIGURE_PATH = ROOT / "reports/figures/model_comparison_metrics.png"
MATRICES_FIGURE_PATH = ROOT / "reports/figures/model_comparison_confusion_matrices.png"
DISAGREEMENTS_PATH = ROOT / "data/processed/model_comparison_disagreements.jsonl"

print({"numpy": np.__version__, "scikit_learn": sklearn.__version__, "scipy": scipy.__version__})


## 3. Validação da comparabilidade

Não basta ler dois relatórios: conferimos que cada linha possui o mesmo chunk, reunião e pseudo-rótulo, na mesma ordem. Se essa condição falhar, o notebook interrompe a execução.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
baseline_report = json.loads(BASELINE_METRICS_PATH.read_text(encoding="utf-8"))
bert_report = json.loads(BERT_METRICS_PATH.read_text(encoding="utf-8"))
baseline_rows = read_jsonl(BASELINE_PREDICTIONS_PATH)
bert_rows = read_jsonl(BERT_PREDICTIONS_PATH)

assert len(baseline_rows) == len(bert_rows) > 0
assert baseline_report["evaluation_reference"] == bert_report["evaluation_reference"]
assert baseline_report["validation_chunks"] == bert_report["validation_chunks"] == len(baseline_rows)
assert baseline_report["validation_meetings"] == bert_report["validation_meetings"]
for baseline_row, bert_row in zip(baseline_rows, bert_rows):
    assert baseline_row["chunk_id"] == bert_row["chunk_id"]
    assert baseline_row["meeting_id"] == bert_row["meeting_id"]
    assert baseline_row["pseudo_label"] == bert_row["pseudo_label"]
    assert "text" not in baseline_row and "text" not in bert_row

meeting_ids = np.asarray([row["meeting_id"] for row in baseline_rows])
chunk_ids = [row["chunk_id"] for row in baseline_rows]
y_true = np.asarray([row["pseudo_label"] for row in baseline_rows], dtype=np.int64)
baseline_predictions = np.asarray([row["prediction"] for row in baseline_rows], dtype=np.int64)
bert_predictions = np.asarray([row["prediction"] for row in bert_rows], dtype=np.int64)
baseline_probabilities = np.asarray([row["probability_oportunidade"] for row in baseline_rows], dtype=np.float64)
bert_probabilities = np.asarray([row["probability_oportunidade"] for row in bert_rows], dtype=np.float64)
unique_meetings = np.unique(meeting_ids)

print({
    "paired_chunks": len(y_true),
    "validation_meetings": len(unique_meetings),
    "same_reference": True,
    "texts_in_prediction_files": False,
})


## 4. Métricas lado a lado

As métricas são recalculadas a partir das previsões pareadas, em vez de apenas copiar valores dos relatórios anteriores.

As métricas são calculadas sobre a mesma referência usada pelo experimento. Precision, recall e F1 complementam a acurácia e ajudam a distinguir o custo de falsos positivos e falsos negativos.


In [ ]:
def calculate_metrics(labels: np.ndarray, predictions: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_oportunidade": float(precision_score(labels, predictions, zero_division=0)),
        "recall_oportunidade": float(recall_score(labels, predictions, zero_division=0)),
        "f1_oportunidade": float(f1_score(labels, predictions, zero_division=0)),
        "f1_macro": float(f1_score(labels, predictions, average="macro", zero_division=0)),
    }


### Preparação dos objetos desta etapa

A célula prepara `baseline_metrics`, `bert_metrics`, `metric_comparison` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
baseline_metrics = calculate_metrics(y_true, baseline_predictions)
bert_metrics = calculate_metrics(y_true, bert_predictions)
metric_comparison = {}
for metric_name in baseline_metrics:
    baseline_value = baseline_metrics[metric_name]
    bert_value = bert_metrics[metric_name]
    if np.isclose(baseline_value, bert_value):
        winner = "empate"
    else:
        winner = "bertimbau" if bert_value > baseline_value else "baseline_tfidf_logreg"
    metric_comparison[metric_name] = {
        "baseline_tfidf_logreg": baseline_value,
        "bertimbau": bert_value,
        "delta_bertimbau_minus_baseline": bert_value - baseline_value,
        "winner": winner,
    }

for metric_name, values in metric_comparison.items():
    print({
        "metric": metric_name,
        "baseline": round(values["baseline_tfidf_logreg"], 4),
        "bertimbau": round(values["bertimbau"], 4),
        "delta": round(values["delta_bertimbau_minus_baseline"], 4),
        "winner": values["winner"],
    })


## 5. Análise pareada dos erros

O teste exato de McNemar usa apenas os casos em que os modelos discordam quanto ao acerto. Com poucos casos discordantes, o valor-p tende a ser pouco conclusivo; ele será registrado, não usado como prova isolada.


In [ ]:
baseline_correct = baseline_predictions == y_true
bert_correct = bert_predictions == y_true
paired_outcomes = {
    "both_correct": int(np.sum(baseline_correct & bert_correct)),
    "baseline_only_correct": int(np.sum(baseline_correct & ~bert_correct)),
    "bertimbau_only_correct": int(np.sum(~baseline_correct & bert_correct)),
    "both_wrong": int(np.sum(~baseline_correct & ~bert_correct)),
}
discordant_total = paired_outcomes["baseline_only_correct"] + paired_outcomes["bertimbau_only_correct"]
mcnemar_pvalue = (
    float(binomtest(paired_outcomes["bertimbau_only_correct"], discordant_total, p=0.5).pvalue)
    if discordant_total else 1.0
)

disagreement_rows = []
for index in np.flatnonzero(baseline_predictions != bert_predictions):
    disagreement_rows.append({
        "meeting_id": str(meeting_ids[index]),
        "chunk_id": chunk_ids[index],
        "pseudo_label": int(y_true[index]),
        "baseline_prediction": int(baseline_predictions[index]),
        "bertimbau_prediction": int(bert_predictions[index]),
        "baseline_correct": bool(baseline_correct[index]),
        "bertimbau_correct": bool(bert_correct[index]),
    })
with DISAGREEMENTS_PATH.open("w", encoding="utf-8") as file:
    for row in disagreement_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print({**paired_outcomes, "discordant_total": discordant_total, "mcnemar_exact_pvalue": round(mcnemar_pvalue, 4)})

## 6. Bootstrap agrupado por reunião

Reamostramos as 98 reuniões com reposição e recalculamos a diferença `BERTimbau − baseline` duas mil vezes. O intervalo de 95% descreve a incerteza neste conjunto; não corrige o viés dos pseudo-rótulos.

O reamostramento é agrupado por reunião para preservar a dependência entre chunks do mesmo encontro. O intervalo resultante mostra a variabilidade da diferença entre modelos, em vez de apresentar apenas uma estimativa pontual.


In [ ]:
indices_by_meeting: dict[str, list[int]] = defaultdict(list)
for index, meeting_id in enumerate(meeting_ids):
    indices_by_meeting[str(meeting_id)].append(index)
meeting_list = sorted(indices_by_meeting)
rng = np.random.default_rng(SEED)
bootstrap_deltas = {metric_name: [] for metric_name in baseline_metrics}

for _ in range(BOOTSTRAP_REPETITIONS):
    sampled_meetings = rng.choice(meeting_list, size=len(meeting_list), replace=True)
    sampled_indices = np.asarray(
        [index for meeting_id in sampled_meetings for index in indices_by_meeting[str(meeting_id)]],
        dtype=np.int64,
    )
    sampled_labels = y_true[sampled_indices]
    sampled_baseline = calculate_metrics(sampled_labels, baseline_predictions[sampled_indices])
    sampled_bert = calculate_metrics(sampled_labels, bert_predictions[sampled_indices])
    for metric_name in bootstrap_deltas:
        bootstrap_deltas[metric_name].append(sampled_bert[metric_name] - sampled_baseline[metric_name])

bootstrap_summary = {}
for metric_name, deltas in bootstrap_deltas.items():
    values = np.asarray(deltas)
    bootstrap_summary[metric_name] = {
        "observed_delta": metric_comparison[metric_name]["delta_bertimbau_minus_baseline"],
        "ci95_low": float(np.quantile(values, 0.025)),
        "ci95_high": float(np.quantile(values, 0.975)),
        "probability_bertimbau_better": float(np.mean(values > 0)),
    }


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
for metric_name, values in bootstrap_summary.items():
    print({
        "metric": metric_name,
        "delta": round(values["observed_delta"], 4),
        "ci95": [round(values["ci95_low"], 4), round(values["ci95_high"], 4)],
        "p_bertimbau_better": round(values["probability_bertimbau_better"], 4),
    })


## 7. Análise para escolha do melhor modelo

Além das classes finais, avaliamos a ordenação e a qualidade das probabilidades. O Brier score mede o erro quadrático das probabilidades e deve ser menor. Também explicitamos o custo relativo dos erros: considerando custo 1 para revisar um falso positivo, o baseline custa 6 e o BERTimbau custa `2 + custo_FN`. O ponto de empate ocorre quando um falso negativo custa quatro vezes uma revisão.

A busca de um limiar com recall total é apenas diagnóstica, pois foi feita na própria validação. O limiar final deverá ser calibrado nos dados humanos.


In [ ]:
probability_quality = {
    "baseline_tfidf_logreg": {
        "roc_auc": float(roc_auc_score(y_true, baseline_probabilities)),
        "average_precision": float(average_precision_score(y_true, baseline_probabilities)),
        "brier_score": float(brier_score_loss(y_true, baseline_probabilities)),
    },
    "bertimbau": {
        "roc_auc": float(roc_auc_score(y_true, bert_probabilities)),
        "average_precision": float(average_precision_score(y_true, bert_probabilities)),
        "brier_score": float(brier_score_loss(y_true, bert_probabilities)),
    },
}


### Função auxiliar: `diagnostic_threshold_for_full_recall`

Esta célula isola a responsabilidade implementada por `diagnostic_threshold_for_full_recall`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def diagnostic_threshold_for_full_recall(probabilities: np.ndarray) -> dict:
    threshold = float(np.min(probabilities[y_true == 1]))
    predictions = (probabilities >= threshold).astype(np.int64)
    matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
    tn_value, fp_value, fn_value, tp_value = (int(value) for value in matrix.ravel())
    return {
        "threshold": threshold,
        "precision_oportunidade": float(precision_score(y_true, predictions, zero_division=0)),
        "recall_oportunidade": float(recall_score(y_true, predictions, zero_division=0)),
        "f1_oportunidade": float(f1_score(y_true, predictions, zero_division=0)),
        "false_positive": fp_value,
        "false_negative": fn_value,
        "warning": "Diagnóstico na validação; não usar como limiar final.",
    }


### Preparação dos objetos desta etapa

A célula prepara `full_recall_diagnostic`, `baseline_fp`, `baseline_fn`, `bert_fp` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
full_recall_diagnostic = {
    "baseline_tfidf_logreg": diagnostic_threshold_for_full_recall(baseline_probabilities),
    "bertimbau": diagnostic_threshold_for_full_recall(bert_probabilities),
}
baseline_fp = int(np.sum((baseline_predictions == 1) & (y_true == 0)))
baseline_fn = int(np.sum((baseline_predictions == 0) & (y_true == 1)))
bert_fp = int(np.sum((bert_predictions == 1) & (y_true == 0)))
bert_fn = int(np.sum((bert_predictions == 0) & (y_true == 1)))
false_negative_cost_break_even = (baseline_fp - bert_fp) / (bert_fn - baseline_fn)
resource_tradeoff = {
    "baseline_fit_seconds": float(baseline_report["fit_seconds"]),
    "bertimbau_training_seconds": float(bert_report["training_seconds"]),
    "baseline_model_size_mb": BASELINE_MODEL_PATH.stat().st_size / 1024**2,
    "bertimbau_model_size_mb": BERT_MODEL_PATH.stat().st_size / 1024**2,
}
best_model_current_experiment = "bertimbau"
selection_reasons = [
    "Venceu em accuracy, precision, F1 de oportunidade e F1 macro.",
    "Cometeu três erros contra seis do baseline no mesmo conjunto.",
    "Acertou sozinho quatro casos, enquanto o baseline acertou sozinho um.",
    "Apresentou Brier score menor, indicando probabilidades mais próximas dos pseudo-rótulos.",
    "O recall pode ser priorizado depois por calibração de limiar em dados humanos.",
]


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
print({
    "best_model_current_experiment": best_model_current_experiment,
    "brier_baseline": round(probability_quality["baseline_tfidf_logreg"]["brier_score"], 4),
    "brier_bertimbau": round(probability_quality["bertimbau"]["brier_score"], 4),
    "false_negative_cost_break_even": false_negative_cost_break_even,
    "baseline_model_mb": round(resource_tradeoff["baseline_model_size_mb"], 2),
    "bertimbau_model_mb": round(resource_tradeoff["bertimbau_model_size_mb"], 2),
})


## 8. Visualizações comparativas

As barras usam escala completa de 0 a 1 para não exagerar pequenas diferenças. As matrizes usam a mesma ordem de classes.

A comparação mantém dados e referência constantes para que a diferença observada seja atribuída aos métodos. Resultados próximos devem ser interpretados com a incerteza estatística e o custo operacional em mente.


In [ ]:
METRICS_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
metric_keys = ["accuracy", "precision_oportunidade", "recall_oportunidade", "f1_oportunidade", "f1_macro"]
metric_labels = ["Accuracy", "Precision\noportunidade", "Recall\noportunidade", "F1\noportunidade", "F1 macro"]
positions = np.arange(len(metric_keys))
width = 0.36
fig, ax = plt.subplots(figsize=(10, 5.5))
baseline_bars = ax.bar(positions - width / 2, [baseline_metrics[key] for key in metric_keys], width, label="TF-IDF + LogReg", color="#4C78A8")
bert_bars = ax.bar(positions + width / 2, [bert_metrics[key] for key in metric_keys], width, label="BERTimbau", color="#7A5195")
ax.bar_label(baseline_bars, fmt="%.3f", padding=3, fontsize=9)
ax.bar_label(bert_bars, fmt="%.3f", padding=3, fontsize=9)
ax.set_xticks(positions, metric_labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Pontuação")
ax.set_title("Comparação na validação com pseudo-rótulos")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(METRICS_FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()

baseline_matrix = confusion_matrix(y_true, baseline_predictions, labels=[0, 1])
bert_matrix = confusion_matrix(y_true, bert_predictions, labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))


### Visualização da matriz de confusão

A matriz apresenta acertos e erros por classe. Ela complementa as métricas agregadas ao mostrar diretamente quais tipos de erro ocorreram.


In [ ]:
for axis, matrix, title, color in [
    (axes[0], baseline_matrix, "TF-IDF + LogReg", "Blues"),
    (axes[1], bert_matrix, "BERTimbau", "Purples"),
]:
    display = ConfusionMatrixDisplay(matrix, display_labels=["Não oportunidade", "Oportunidade"])
    display.plot(ax=axis, cmap=color, values_format="d", colorbar=False)
    axis.set_title(title)
    axis.set_xlabel("Rótulo predito")
    axis.set_ylabel("Pseudo-rótulo")
fig.suptitle("Matrizes de confusão no mesmo conjunto de validação", y=1.02)
fig.tight_layout()
fig.savefig(MATRICES_FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()


## 9. Melhor modelo do experimento e relatório

**Escolha provisória: BERTimbau.** Ele vence quatro das cinco métricas principais, reduz os erros de seis para três e produz probabilidades com Brier score muito menor. O baseline continua sendo a referência mais barata e obteve recall 100% no limiar padrão. Se o negócio definir que um falso negativo custa pelo menos quatro vezes uma revisão, o baseline passa a ter menor custo neste conjunto. A escolha do BERTimbau é a melhor para o experimento atual, mas ainda precisa ser confirmada na auditoria humana.


In [ ]:
report = {
    "schema_version": "1.0",
    "evaluation_reference": baseline_report["evaluation_reference"],
    "warning": "Comparação contra pseudo-rótulos; a decisão final exige o conjunto humano reservado.",
    "paired_chunks": len(y_true),
    "validation_meetings": len(unique_meetings),
    "same_split_verified": True,
    "metrics": metric_comparison,
    "confusion_matrices": {
        "order": [["TN", "FP"], ["FN", "TP"]],
        "baseline_tfidf_logreg": baseline_matrix.tolist(),
        "bertimbau": bert_matrix.tolist(),
    },
    "paired_outcomes": paired_outcomes,
    "probability_quality": probability_quality,
    "full_recall_threshold_diagnostic": full_recall_diagnostic,
    "operational_cost_analysis": {
        "false_positive_cost": 1.0,
        "false_negative_cost_break_even": false_negative_cost_break_even,
        "interpretation": "BERTimbau tem menor custo se FN custar menos de 4 revisões; baseline tem menor custo se custar mais de 4.",
    },
    "resource_tradeoff": resource_tradeoff,
    "mcnemar_exact": {
        "discordant_total": discordant_total,
        "pvalue_two_sided": mcnemar_pvalue,
        "interpretation": "A diferença de acertos não é conclusiva com esta amostra se p >= 0.05.",
    },
    "grouped_bootstrap": {
        "unit": "meeting_id",
        "repetitions": BOOTSTRAP_REPETITIONS,
        "seed": SEED,
        "delta_order": "bertimbau_minus_baseline",
        "metrics": bootstrap_summary,
    },
    "decision": {
        "primary_metric": "recall_oportunidade",
        "primary_metric_justification": "Falsos negativos podem impedir que oportunidades comerciais cheguem ao time responsável.",
        "primary_metric_winner_on_pseudo_labels": "baseline_tfidf_logreg",
        "balanced_performance_winner_on_pseudo_labels": "bertimbau",
        "best_model_current_experiment": best_model_current_experiment,
        "selection_status": "provisional_until_human_evaluation",
        "selection_reasons": selection_reasons,
        "final_model_selected": None,
        "next_gate": "Rotular a auditoria humana e repetir a comparação em reuniões nunca usadas no desenvolvimento.",
    },
}


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print({
    "primary_metric": report["decision"]["primary_metric"],
    "primary_metric_winner": report["decision"]["primary_metric_winner_on_pseudo_labels"],
    "balanced_winner": report["decision"]["balanced_performance_winner_on_pseudo_labels"],
    "best_model_current_experiment": report["decision"]["best_model_current_experiment"],
    "final_model_selected": report["decision"]["final_model_selected"],
    "next_gate": report["decision"]["next_gate"],
})


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 9 de 11 — 09 rag retrieval evolution

Esta seção reproduz a etapa `09_rag_retrieval_evolution.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 09 — Evolução e avaliação do retrieval do RAG

Este notebook transforma a busca lexical mínima em um experimento mensurável. Ele compara BM25, embeddings com BERTimbau, busca híbrida por Reciprocal Rank Fusion (RRF) e uma versão com reranking leve por nível de evidência.

A avaliação usa 32 consultas curadas com documentos relevantes esperados. Os resultados medem retrieval, não a qualidade de uma resposta gerada por LLM.

**Objetivo e conexão com o projeto.** Esta etapa evolui a recuperação da base TOTVS e compara abordagens lexicais, vetoriais e híbridas sobre consultas com documentos esperados. A avaliação não observa apenas relevância. O contexto recuperado também precisa carregar fonte, nível de evidência e política de uso para que hipóteses não sejam apresentadas como fatos.


## 1. Decisões

- Stopwords em português são removidas apenas do índice lexical; o conteúdo original não é alterado.
- Título e produto recebem peso 3, palavras-chave peso 2 e conteúdo peso 1 no BM25.
- Aliases são aplicados tanto às consultas quanto aos documentos.
- O retriever vetorial usa mean pooling do BERTimbau base; ele é um baseline de embeddings, não um modelo especializado em similaridade de sentenças.
- A busca híbrida combina posições do BM25 e dos embeddings por RRF.
- O reranking usa um bônus pequeno para evidência mais forte e presença de fontes, sem permitir que esse bônus domine relevância.
- Recall@k, Hit@k, MRR e nDCG@k são calculados de forma reproduzível.
- Resultados destinados a contexto carregam fontes e uma política explícita para fatos, hipóteses e recomendações.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import tempfile
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "wedjat-matplotlib"))
warnings.filterwarnings("ignore", message="IProgress not found.*")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 512
RRF_K = 60
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2


### Função auxiliar: `find_project_root`

Esta célula isola a responsabilidade implementada por `find_project_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada. No Colab, entre na pasta Wedjat.")


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `KB_PATH`, `ALIASES_PATH`, `QUERIES_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT = find_project_root()
KB_PATH = ROOT / "data/knowledge_base/totvs_rag_kb_v1.json"
ALIASES_PATH = ROOT / "data/knowledge_base/rag_aliases.json"
QUERIES_PATH = ROOT / "data/knowledge_base/rag_evaluation_queries.json"
EMBEDDINGS_PATH = ROOT / "data/processed/rag_bertimbau_embeddings.npz"
REPORT_PATH = ROOT / "reports/metrics/rag_retrieval_evaluation.json"
KEYWORD_AUDIT_PATH = ROOT / "reports/metrics/rag_keyword_audit.json"
FIGURE_PATH = ROOT / "reports/figures/rag_retrieval_comparison.png"

print({
    "device": str(DEVICE),
    "device_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
    "torch": torch.__version__,
    "transformers": transformers.__version__,
})


## 3. Carga e validação dos ativos

A execução é interrompida se houver IDs duplicados, consultas sem documento esperado ou aliases repetidos em grupos diferentes.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
knowledge_base = json.loads(KB_PATH.read_text(encoding="utf-8"))
aliases_payload = json.loads(ALIASES_PATH.read_text(encoding="utf-8"))
queries_payload = json.loads(QUERIES_PATH.read_text(encoding="utf-8"))
alias_groups = aliases_payload["groups"]
evaluation_queries = queries_payload["queries"]
document_ids = [document["id"] for document in knowledge_base]
document_id_set = set(document_ids)

assert len(document_ids) == len(document_id_set), "IDs duplicados na base."
assert len({query["id"] for query in evaluation_queries}) == len(evaluation_queries)
assert all(query["expected_ids"] for query in evaluation_queries)
assert all(set(query["expected_ids"]) <= document_id_set for query in evaluation_queries)
all_aliases = [alias.casefold() for group in alias_groups for alias in [group["canonical"], *group["aliases"]]]
assert len(all_aliases) == len(set(all_aliases)), "Alias repetido em mais de um grupo."

print({
    "documents": len(knowledge_base),
    "evaluation_queries": len(evaluation_queries),
    "alias_groups": len(alias_groups),
    "expected_links": sum(len(query["expected_ids"]) for query in evaluation_queries),
})

## 4. Normalização, auditoria de palavras-chave e aliases

A auditoria identifica palavras-chave vazias, formadas só por stopwords ou frequentes demais. O relatório é agregado; nenhuma alteração automática é feita na base original.

A auditoria usa apenas estrutura e valores agregados. Essa escolha permite avaliar cobertura e distribuição sem imprimir transcrições ou outros conteúdos que não devem aparecer no material entregue.


In [ ]:
PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "aquela", "aquele", "aqueles", "as", "até", "com", "como",
    "da", "das", "de", "dela", "dele", "do", "dos", "e", "ela", "elas", "ele", "eles",
    "em", "entre", "era", "essa", "esse", "esta", "este", "eu", "foi", "isso", "isto",
    "já", "mais", "mas", "me", "mesmo", "meu", "minha", "muito", "na", "não", "nas",
    "no", "nos", "nós", "o", "os", "ou", "para", "pela", "pelo", "por", "qual",
    "que", "se", "sem", "ser", "seu", "sua", "são", "também", "tem", "um", "uma",
    "você", "vs", "x"
}


### Função auxiliar: `normalize_text`

Esta célula isola a responsabilidade implementada por `normalize_text`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def normalize_text(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(text).casefold())
    return "".join(character for character in normalized if not unicodedata.combining(character))


### Preparação dos objetos desta etapa

A célula prepara `normalized_alias_groups` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
normalized_alias_groups = []
for group in alias_groups:
    variants = [group["canonical"], *group["aliases"]]
    normalized_alias_groups.append([normalize_text(variant) for variant in variants])


### Função auxiliar: `tokenize`

Esta célula isola a responsabilidade implementada por `tokenize`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def tokenize(text: str, expand_aliases: bool = True) -> list[str]:
    normalized = normalize_text(text)
    tokens = [token for token in re.findall(r"[a-z0-9+]+", normalized) if len(token) >= 2 and token not in PORTUGUESE_STOPWORDS]
    if expand_aliases:
        padded = f" {normalized} "
        for variants in normalized_alias_groups:
            if any(f" {variant} " in padded for variant in variants):
                expansion = " ".join(variants)
                tokens.extend(tokenize(expansion, expand_aliases=False))
    return tokens


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [ ]:
keyword_document_frequency = Counter()
empty_or_stopword_keywords = set()
for document in knowledge_base:
    normalized_keywords = set()
    for keyword in document.get("keywords", []):
        keyword_tokens = tokenize(keyword, expand_aliases=False)
        if not keyword_tokens:
            empty_or_stopword_keywords.add(str(keyword))
        else:
            normalized_keywords.add(" ".join(keyword_tokens))
    keyword_document_frequency.update(normalized_keywords)
common_threshold = math.ceil(len(knowledge_base) * 0.20)
overly_common_keywords = {
    keyword: frequency for keyword, frequency in keyword_document_frequency.items() if frequency >= common_threshold
}
keyword_audit = {
    "documents": len(knowledge_base),
    "unique_normalized_keywords": len(keyword_document_frequency),
    "common_threshold_documents": common_threshold,
    "empty_or_stopword_keywords": sorted(empty_or_stopword_keywords),
    "overly_common_keywords": dict(sorted(overly_common_keywords.items(), key=lambda item: (-item[1], item[0]))),
    "most_frequent_keywords": keyword_document_frequency.most_common(20),
    "decision": "Stopwords são ignoradas no índice; keywords originais permanecem intactas.",
}
KEYWORD_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
KEYWORD_AUDIT_PATH.write_text(json.dumps(keyword_audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
print({
    "unique_keywords": len(keyword_document_frequency),
    "empty_or_stopword": len(empty_or_stopword_keywords),
    "overly_common": len(overly_common_keywords),
})


## 5. Retriever lexical BM25

BM25 considera frequência do termo, raridade na coleção e comprimento do documento. Repetimos campos conforme seus pesos antes de construir o índice.

O retriever ordena documentos candidatos para uma consulta. Esta implementação é avaliada separadamente para distinguir a qualidade da recuperação da qualidade do classificador de oportunidades.


In [ ]:
def document_text(document: dict) -> str:
    fields = [
        document.get("title", ""),
        document.get("product", ""),
        document.get("category", ""),
        " ".join(document.get("segments", [])),
        " ".join(document.get("keywords", [])),
        " ".join(document.get("related_products", [])),
        " ".join(document.get("competitors", [])),
        document.get("content", ""),
    ]
    return ". ".join(str(field) for field in fields if field)

def weighted_document_tokens(document: dict) -> list[str]:
    weighted_fields = [
        (document.get("title", ""), 3),
        (document.get("product", ""), 3),
        (" ".join(document.get("keywords", [])), 2),
        (document.get("category", ""), 1),
        (" ".join(document.get("segments", [])), 1),
        (" ".join(document.get("related_products", [])), 1),
        (" ".join(document.get("competitors", [])), 2),
        (document.get("content", ""), 1),
    ]
    result = []
    for value, weight in weighted_fields:
        result.extend(tokenize(value) * weight)
    return result


### Preparação dos objetos desta etapa

A célula prepara `bm25_document_tokens`, `bm25_term_frequencies`, `bm25_lengths`, `bm25_average_length` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
bm25_document_tokens = [weighted_document_tokens(document) for document in knowledge_base]
bm25_term_frequencies = [Counter(tokens) for tokens in bm25_document_tokens]
bm25_lengths = np.asarray([len(tokens) for tokens in bm25_document_tokens], dtype=np.float64)
bm25_average_length = float(np.mean(bm25_lengths))
bm25_document_frequency = Counter()
for tokens in bm25_document_tokens:
    bm25_document_frequency.update(set(tokens))


### Funções auxiliares da seção

Esta célula agrupa `bm25_scores`, `rank_from_scores`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> np.ndarray:
    query_terms = tokenize(query)
    scores = np.zeros(len(knowledge_base), dtype=np.float64)
    for term in query_terms:
        document_frequency = bm25_document_frequency.get(term, 0)
        if document_frequency == 0:
            continue
        idf = math.log(1.0 + (len(knowledge_base) - document_frequency + 0.5) / (document_frequency + 0.5))
        for index, frequencies in enumerate(bm25_term_frequencies):
            frequency = frequencies.get(term, 0)
            if frequency:
                denominator = frequency + k1 * (1.0 - b + b * bm25_lengths[index] / bm25_average_length)
                scores[index] += idf * frequency * (k1 + 1.0) / denominator
    return scores

def rank_from_scores(scores: np.ndarray) -> list[int]:
    return np.argsort(-scores, kind="stable").tolist()


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
print({"indexed_documents": len(bm25_document_tokens), "bm25_terms": len(bm25_document_frequency)})


## 6. Retriever vetorial com BERTimbau

Os vetores são normalizados e comparados por produto escalar, equivalente à similaridade de cosseno. O arquivo `.npz` fica em `data/processed/` e não é versionado.

O retriever ordena documentos candidatos para uma consulta. Esta implementação é avaliada separadamente para distinguir a qualidade da recuperação da qualidade do classificador de oportunidades.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
embedding_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
embedding_model.eval()


### Funções auxiliares da seção

Esta célula agrupa `expand_text_with_aliases`, `embed_texts`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def expand_text_with_aliases(text: str) -> str:
    normalized = normalize_text(text)
    additions = []
    padded = f" {normalized} "
    for group, variants in zip(alias_groups, normalized_alias_groups):
        if any(f" {variant} " in padded for variant in variants):
            additions.append(" ".join([group["canonical"], *group["aliases"]]))
    return text if not additions else text + ". Aliases: " + ". ".join(additions)

def embed_texts(texts: list[str]) -> tuple[np.ndarray, float]:
    vectors = []
    started = time.perf_counter()
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                hidden = embedding_model(**encoded).last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1)
            pooled = (hidden.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled = F.normalize(pooled, p=2, dim=1)
        vectors.append(pooled.cpu().numpy())
    return np.concatenate(vectors, axis=0), time.perf_counter() - started


### Preparação dos objetos desta etapa

A célula prepara `dense_document_texts`, `dense_score_matrix` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
dense_document_texts = [expand_text_with_aliases(document_text(document)) for document in knowledge_base]
document_embeddings, document_embedding_seconds = embed_texts(dense_document_texts)
query_embeddings, query_embedding_seconds = embed_texts([expand_text_with_aliases(item["query"]) for item in evaluation_queries])
np.savez_compressed(EMBEDDINGS_PATH, document_ids=np.asarray(document_ids), embeddings=document_embeddings)
dense_score_matrix = query_embeddings @ document_embeddings.T

print({
    "embedding_dimensions": int(document_embeddings.shape[1]),
    "document_embedding_seconds": round(document_embedding_seconds, 3),
    "query_embedding_seconds": round(query_embedding_seconds, 3),
})


## 7. Fusão híbrida e reranking por evidência

RRF combina posições, evitando comparar diretamente escalas incompatíveis de BM25 e cosseno. O bônus de evidência é deliberadamente pequeno.


In [ ]:
def reciprocal_rank_fusion(rankings: list[list[int]], rrf_k: int = RRF_K) -> np.ndarray:
    scores = np.zeros(len(knowledge_base), dtype=np.float64)
    for ranking in rankings:
        for position, document_index in enumerate(ranking, start=1):
            scores[document_index] += 1.0 / (rrf_k + position)
    return scores

def evidence_priority(document: dict) -> float:
    level = normalize_text(document.get("evidence_level", ""))
    if "fato documentado" in level:
        return 1.0
    if "fato conceitual" in level:
        return 0.9
    if "fato" in level and "inferencia" in level:
        return 0.7
    if "hipotese" in level:
        return 0.4
    if "recomendacao" in level or "validacao" in level:
        return 0.3
    return 0.5


### Preparação dos objetos desta etapa

A célula prepara `retriever_rankings` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
bm25_rankings, dense_rankings, hybrid_rankings, reranked_rankings = [], [], [], []
for query_index, query_item in enumerate(evaluation_queries):
    bm25_ranking = rank_from_scores(bm25_scores(query_item["query"]))
    dense_ranking = rank_from_scores(dense_score_matrix[query_index])
    hybrid_scores = reciprocal_rank_fusion([bm25_ranking, dense_ranking])
    reranked_scores = hybrid_scores.copy()
    for index, document in enumerate(knowledge_base):
        reranked_scores[index] += 0.00025 * evidence_priority(document)
        reranked_scores[index] += 0.00010 * bool(document.get("sources"))
    bm25_rankings.append(bm25_ranking)
    dense_rankings.append(dense_ranking)
    hybrid_rankings.append(rank_from_scores(hybrid_scores))
    reranked_rankings.append(rank_from_scores(reranked_scores))

retriever_rankings = {
    "bm25_aliases": bm25_rankings,
    "bertimbau_embeddings": dense_rankings,
    "hybrid_rrf": hybrid_rankings,
    "hybrid_evidence_rerank": reranked_rankings,
}
print({name: len(rankings) for name, rankings in retriever_rankings.items()})


## 8. Avaliação dos retrievers

Recall@k mede a fração dos documentos relevantes recuperada; Hit@k indica se ao menos um relevante apareceu; MRR privilegia o primeiro relevante em posição alta; nDCG considera todos os relevantes e sua ordem.

A avaliação transforma as previsões em medidas comparáveis e mantém a unidade de análise explícita. Como os rótulos atuais são automáticos, os números descrevem o experimento de desenvolvimento e não desempenho comprovado em produção.


In [ ]:
def evaluate_rankings(rankings: list[list[int]]) -> tuple[dict, list[dict]]:
    cutoffs = (1, 3, 5)
    recall_values = {cutoff: [] for cutoff in cutoffs}
    hit_values = {cutoff: [] for cutoff in cutoffs}
    ndcg_values = {cutoff: [] for cutoff in cutoffs}
    reciprocal_ranks = []
    details = []
    for query_item, ranking in zip(evaluation_queries, rankings):
        ranked_ids = [document_ids[index] for index in ranking]
        relevant = set(query_item["expected_ids"])
        relevant_positions = [position for position, document_id in enumerate(ranked_ids, start=1) if document_id in relevant]
        reciprocal_rank = 1.0 / min(relevant_positions) if relevant_positions else 0.0
        reciprocal_ranks.append(reciprocal_rank)
        query_metrics = {"query_id": query_item["id"], "expected_ids": sorted(relevant), "first_relevant_rank": min(relevant_positions) if relevant_positions else None, "top5_ids": ranked_ids[:5]}
        for cutoff in cutoffs:
            top_ids = ranked_ids[:cutoff]
            gains = [1 if document_id in relevant else 0 for document_id in top_ids]
            recall = sum(gains) / len(relevant)
            hit = float(any(gains))
            dcg = sum(gain / math.log2(position + 2) for position, gain in enumerate(gains))
            ideal_gains = [1] * min(len(relevant), cutoff)
            idcg = sum(gain / math.log2(position + 2) for position, gain in enumerate(ideal_gains))
            recall_values[cutoff].append(recall)
            hit_values[cutoff].append(hit)
            ndcg_values[cutoff].append(dcg / idcg if idcg else 0.0)
            query_metrics[f"recall@{cutoff}"] = recall
        details.append(query_metrics)
    aggregate = {"mrr": float(np.mean(reciprocal_ranks))}
    for cutoff in cutoffs:
        aggregate[f"recall@{cutoff}"] = float(np.mean(recall_values[cutoff]))
        aggregate[f"hit@{cutoff}"] = float(np.mean(hit_values[cutoff]))
        aggregate[f"ndcg@{cutoff}"] = float(np.mean(ndcg_values[cutoff]))
    return aggregate, details


### Preparação dos objetos desta etapa

A célula prepara `best_retriever` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
retrieval_metrics, query_details = {}, {}
for name, rankings in retriever_rankings.items():
    retrieval_metrics[name], query_details[name] = evaluate_rankings(rankings)
    print(name, {key: round(value, 4) for key, value in retrieval_metrics[name].items()})

best_retriever = max(retrieval_metrics, key=lambda name: (retrieval_metrics[name]["recall@5"], retrieval_metrics[name]["mrr"], retrieval_metrics[name]["ndcg@5"]))
print({"best_retriever_by_recall5_then_mrr": best_retriever})


## 9. Contrato de contexto fundamentado

O retrieval não deve apagar o nível de evidência. Cada resultado carrega URLs e uma política que a futura camada geradora deverá obedecer. Documentos sem fontes não podem sustentar afirmações factuais.


In [ ]:
def claim_policy(document: dict) -> str:
    level = normalize_text(document.get("evidence_level", ""))
    if not document.get("sources"):
        return "context_only_do_not_state_as_fact_without_source"
    if "hipotese" in level:
        return "state_only_as_hypothesis_with_citation"
    if "recomendacao" in level or "validacao" in level:
        return "state_as_recommendation_or_pending_validation_with_citation"
    return "factual_claim_requires_citation"

def retrieve_grounded(query: str, top_k: int = 5) -> list[dict]:
    bm25_ranking = rank_from_scores(bm25_scores(query))
    query_vector, _ = embed_texts([expand_text_with_aliases(query)])
    dense_ranking = rank_from_scores(query_vector @ document_embeddings.T)
    scores = reciprocal_rank_fusion([bm25_ranking, dense_ranking])
    for index, document in enumerate(knowledge_base):
        scores[index] += 0.00025 * evidence_priority(document)
        scores[index] += 0.00010 * bool(document.get("sources"))
    results = []
    for index in rank_from_scores(scores)[:top_k]:
        document = knowledge_base[index]
        results.append({
            "id": document["id"],
            "title": document["title"],
            "evidence_level": document["evidence_level"],
            "source_urls": [source["url"] for source in document.get("sources", [])],
            "claim_policy": claim_policy(document),
        })
    return results


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
grounding_validation = [retrieve_grounded(query_item["query"], top_k=5) for query_item in evaluation_queries[:3]]
assert all("claim_policy" in result and "source_urls" in result for results in grounding_validation for result in results)
assert all(result["source_urls"] or result["claim_policy"].startswith("context_only") for results in grounding_validation for result in results)
print({"grounded_queries_validated": len(grounding_validation), "policy_contract_ok": True})


## 10. Relatório e visualização

O relatório registra métricas agregadas e rankings por consulta. As consultas e a base são públicas no repositório; nenhuma transcrição de reunião participa desta etapa.

A visualização resume os resultados já calculados e facilita a comparação entre alternativas. A escala e os rótulos devem ser lidos junto das métricas numéricas, sem ampliar visualmente diferenças pequenas.


In [ ]:
metric_names = ["recall@1", "recall@3", "recall@5", "mrr", "ndcg@5"]
retriever_names = list(retrieval_metrics)
positions = np.arange(len(metric_names))
width = 0.19
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#4C78A8", "#F58518", "#54A24B", "#7A5195"]
for index, (name, color) in enumerate(zip(retriever_names, colors)):
    offset = (index - (len(retriever_names) - 1) / 2) * width
    values = [retrieval_metrics[name][metric] for metric in metric_names]
    bars = ax.bar(positions + offset, values, width, label=name, color=color)
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8, rotation=90)
ax.set_xticks(positions, metric_names)
ax.set_ylim(0, 1.13)
ax.set_ylabel("Pontuação")
ax.set_title("Comparação dos retrievers em 32 consultas curadas")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout(rect=(0, 0.08, 1, 1))
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()


### Preparação dos objetos desta etapa

A célula prepara `report` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
report = {
    "schema_version": "1.0",
    "knowledge_base": str(KB_PATH.relative_to(ROOT)),
    "documents": len(knowledge_base),
    "evaluation_queries": len(evaluation_queries),
    "query_set_warning": queries_payload["warning"],
    "aliases_file": str(ALIASES_PATH.relative_to(ROOT)),
    "alias_groups": len(alias_groups),
    "retrievers": {
        "bm25_aliases": {"description": "BM25 com campos ponderados, stopwords e aliases"},
        "bertimbau_embeddings": {"description": "Mean pooling do BERTimbau base e similaridade de cosseno"},
        "hybrid_rrf": {"description": "Fusão BM25 + embeddings por RRF", "rrf_k": RRF_K},
        "hybrid_evidence_rerank": {"description": "RRF com bônus pequeno por evidência e fontes"},
    },
    "metrics": retrieval_metrics,
    "best_retriever_by_recall5_then_mrr": best_retriever,
    "query_details": query_details,
    "embedding": {
        "checkpoint": MODEL_NAME,
        "pooling": "attention-mask mean pooling",
        "dimensions": int(document_embeddings.shape[1]),
        "max_length": MAX_LENGTH,
        "device": str(DEVICE),
        "document_embedding_seconds": document_embedding_seconds,
        "query_embedding_seconds": query_embedding_seconds,
    },
    "keyword_audit": str(KEYWORD_AUDIT_PATH.relative_to(ROOT)),
    "grounding_contract": {
        "sources_are_returned": True,
        "hypotheses_have_explicit_policy": True,
        "documents_without_sources_cannot_support_facts": True,
    },
}


### Persistência dos artefatos

Os resultados desta etapa são gravados nos caminhos definidos para consumo posterior. O conteúdo persistido segue o contrato já estabelecido pelo projeto.


In [ ]:
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print({
    "best_retriever": best_retriever,
    "saved_report": str(REPORT_PATH.relative_to(ROOT)),
    "saved_figure": str(FIGURE_PATH.relative_to(ROOT)),
    "saved_keyword_audit": str(KEYWORD_AUDIT_PATH.relative_to(ROOT)),
})


## 11. Próximas evoluções

O melhor retriever desta rodada deve ser mantido como referência. Ainda precisamos revisar manualmente o conjunto de consultas, experimentar um modelo especializado em embeddings, separar fatos de inferências nos conteúdos mistos e adicionar datas reais de verificação das fontes antes de construir a geração final de respostas.

Os itens a seguir separam o que já foi demonstrado do que ainda depende de dados humanos, calibração ou evolução técnica. Assim, o notebook termina sem transformar resultados experimentais em garantias de produção.


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 10 de 11 — 10 sentence embeddings retrieval

Esta seção reproduz a etapa `10_sentence_embeddings_retrieval.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 10 — Embeddings especializados para o RAG

Este notebook testa um encoder treinado para recuperação semântica e o compara, no mesmo conjunto de 32 consultas, com o BM25 e os resultados do notebook 09.

**Decisões:** usamos `intfloat/multilingual-e5-small`, prefixos `query:`/`passage:`, mean pooling com máscara e normalização L2. O modelo é pequeno o bastante para a RTX 3050 e o Colab. A seleção continua priorizando `Recall@5`, seguida de MRR e nDCG@5. O conjunto é pequeno e curado a partir da própria base; portanto o resultado é provisório.

**Objetivo e conexão com o projeto.** O experimento substitui o encoder genérico por um modelo treinado especificamente para recuperação semântica e o compara com os retrievers anteriores. Consultas e documentos recebem os prefixos esperados pelo E5; a escolha do vencedor usa o mesmo conjunto de avaliação, preservando a comparabilidade com a etapa 09.


## 1. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations
import json, math, os, re, tempfile, time, unicodedata, warnings
from collections import Counter
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'wedjat-matplotlib'))
warnings.filterwarnings('ignore', message='IProgress not found.*')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
SEED=42; MODEL_NAME='intfloat/multilingual-e5-small'; MAX_LENGTH=512; RRF_K=60
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=16 if DEVICE.type=='cuda' else 4


### Função auxiliar: `find_root`

Esta célula isola a responsabilidade implementada por `find_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'data').exists() and (p/'notebooks').exists(): return p
    raise FileNotFoundError('Raiz do projeto não encontrada.')


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `KB_PATH`, `ALIASES_PATH`, `QUERIES_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT=find_root(); KB_PATH=ROOT/'data/knowledge_base/totvs_rag_kb_v1.json'
ALIASES_PATH=ROOT/'data/knowledge_base/rag_aliases.json'; QUERIES_PATH=ROOT/'data/knowledge_base/rag_evaluation_queries.json'
PREVIOUS_PATH=ROOT/'reports/metrics/rag_retrieval_evaluation.json'
REPORT_PATH=ROOT/'reports/metrics/sentence_embeddings_retrieval.json'
EMBEDDINGS_PATH=ROOT/'data/processed/rag_multilingual_e5_small_embeddings.npz'
FIGURE_PATH=ROOT/'reports/figures/sentence_embeddings_retrieval.png'
print({'device':str(DEVICE),'device_name':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU','model':MODEL_NAME})


## 2. Dados e baseline lexical
A implementação lexical é mantida igual à do notebook 09 para que a comparação seja pareada e reproduzível.

Antes de transformar ou modelar, conferimos os dados usados nesta etapa e as condições que garantem comparabilidade. As verificações protegem a sequência do notebook contra arquivos incompletos e partições incompatíveis.


In [ ]:
kb=json.loads(KB_PATH.read_text(encoding='utf-8')); aliases=json.loads(ALIASES_PATH.read_text(encoding='utf-8'))['groups']
queries=json.loads(QUERIES_PATH.read_text(encoding='utf-8'))['queries']; previous=json.loads(PREVIOUS_PATH.read_text(encoding='utf-8'))
doc_ids=[d['id'] for d in kb]; assert all(set(q['expected_ids'])<=set(doc_ids) for q in queries)
STOP={'a','ao','aos','as','com','como','da','das','de','do','dos','e','em','entre','essa','esse','esta','este','foi','isso','mais','mas','muito','na','não','nas','no','nos','o','os','ou','para','pela','pelo','por','qual','que','se','sem','ser','sua','são','tem','um','uma','você','vs'}


### Função auxiliar: `norm`

Esta célula isola a responsabilidade implementada por `norm`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def norm(s):
    s=unicodedata.normalize('NFKD',str(s).casefold()); return ''.join(c for c in s if not unicodedata.combining(c))


### Preparação dos objetos desta etapa

A célula prepara `alias_variants` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
alias_variants=[[norm(x) for x in [g['canonical'],*g['aliases']]] for g in aliases]


### Funções auxiliares da seção

Esta célula agrupa `tokens`, `doc_text`, `weighted_tokens`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def tokens(text, expand=True):
    n=norm(text); out=[x for x in re.findall(r'[a-z0-9+]+',n) if len(x)>=2 and x not in STOP]
    if expand:
        padded=f' {n} '
        for variants in alias_variants:
            if any(f' {v} ' in padded for v in variants): out.extend(tokens(' '.join(variants),False))
    return out
def doc_text(d):
    values=[d.get('title',''),d.get('product',''),d.get('category',''),' '.join(d.get('segments',[])),' '.join(d.get('keywords',[])),' '.join(d.get('related_products',[])),' '.join(d.get('competitors',[])),d.get('content','')]
    return '. '.join(str(x) for x in values if x)
def weighted_tokens(d):
    fields=[(d.get('title',''),3),(d.get('product',''),3),(' '.join(d.get('keywords',[])),2),(d.get('category',''),1),(' '.join(d.get('segments',[])),1),(' '.join(d.get('related_products',[])),1),(' '.join(d.get('competitors',[])),2),(d.get('content',''),1)]
    return [t for value,w in fields for t in tokens(value)*w]


### Preparação dos objetos desta etapa

A célula prepara `doc_tokens`, `tf`, `lengths`, `avg` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
doc_tokens=[weighted_tokens(d) for d in kb]; tf=[Counter(x) for x in doc_tokens]; lengths=np.array([len(x) for x in doc_tokens],float); avg=lengths.mean(); df=Counter()
for x in doc_tokens: df.update(set(x))


### Funções auxiliares da seção

Esta célula agrupa `bm25`, `rank`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def bm25(query,k1=1.5,b=.75):
    scores=np.zeros(len(kb))
    for term in tokens(query):
        freq_docs=df.get(term,0)
        if not freq_docs: continue
        idf=math.log(1+(len(kb)-freq_docs+.5)/(freq_docs+.5))
        for i,freqs in enumerate(tf):
            f=freqs.get(term,0)
            if f: scores[i]+=idf*f*(k1+1)/(f+k1*(1-b+b*lengths[i]/avg))
    return scores
def rank(scores): return np.argsort(-scores,kind='stable').tolist()


### Preparação dos objetos desta etapa

A célula prepara `bm25_rankings` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
bm25_rankings=[rank(bm25(q['query'])) for q in queries]
print({'documents':len(kb),'queries':len(queries),'aliases':len(aliases)})


## 3. Embeddings E5
Consultas e documentos recebem prefixos diferentes, conforme o objetivo contrastivo do encoder. Não fazemos ajuste fino com as 32 consultas, evitando otimizar diretamente sobre o conjunto de avaliação.


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME); model=AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()


### Função auxiliar: `embed`

Esta célula isola a responsabilidade implementada por `embed`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def embed(texts,prefix):
    vectors=[]; started=time.perf_counter()
    for start in range(0,len(texts),BATCH_SIZE):
        batch=[prefix+x for x in texts[start:start+BATCH_SIZE]]
        encoded=tokenizer(batch,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
        with torch.inference_mode(), torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda'):
            hidden=model(**encoded).last_hidden_state
        mask=encoded['attention_mask'].unsqueeze(-1); pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1)
        vectors.append(F.normalize(pooled,p=2,dim=1).cpu().numpy())
    return np.concatenate(vectors),time.perf_counter()-started


### Preparação dos objetos desta etapa

A célula prepara `dense_rankings` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
doc_embeddings,doc_seconds=embed([doc_text(d) for d in kb],'passage: ')
query_embeddings,query_seconds=embed([q['query'] for q in queries],'query: ')
EMBEDDINGS_PATH.parent.mkdir(parents=True,exist_ok=True); np.savez_compressed(EMBEDDINGS_PATH,document_ids=np.array(doc_ids),embeddings=doc_embeddings)
dense_rankings=[rank(scores) for scores in query_embeddings@doc_embeddings.T]


### Função auxiliar: `rrf`

Esta célula isola a responsabilidade implementada por `rrf`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def rrf(*rankings):
    scores=np.zeros(len(kb))
    for ranking in rankings:
        for pos,i in enumerate(ranking,1): scores[i]+=1/(RRF_K+pos)
    return scores


### Preparação dos objetos desta etapa

A célula prepara `hybrid_rankings` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
hybrid_rankings=[rank(rrf(b,d)) for b,d in zip(bm25_rankings,dense_rankings)]
print({'dimensions':doc_embeddings.shape[1],'document_seconds':round(doc_seconds,3),'query_seconds':round(query_seconds,3)})


## 4. Avaliação, decisão e artefatos

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


In [ ]:
def evaluate(rankings):
    vals={k:[] for k in ['mrr','recall@1','hit@1','ndcg@1','recall@3','hit@3','ndcg@3','recall@5','hit@5','ndcg@5']}; details=[]
    for q,r in zip(queries,rankings):
        ids=[doc_ids[i] for i in r]; relevant=set(q['expected_ids']); positions=[i for i,x in enumerate(ids,1) if x in relevant]
        vals['mrr'].append(1/min(positions) if positions else 0); row={'query_id':q['id'],'expected_ids':sorted(relevant),'first_relevant_rank':min(positions) if positions else None,'top5_ids':ids[:5]}
        for k in (1,3,5):
            gains=[int(x in relevant) for x in ids[:k]]; recall=sum(gains)/len(relevant); dcg=sum(g/math.log2(i+2) for i,g in enumerate(gains)); ideal=sum(1/math.log2(i+2) for i in range(min(k,len(relevant))))
            vals[f'recall@{k}'].append(recall); vals[f'hit@{k}'].append(float(any(gains))); vals[f'ndcg@{k}'].append(dcg/ideal if ideal else 0); row[f'recall@{k}']=recall
        details.append(row)
    return {k:float(np.mean(v)) for k,v in vals.items()},details


### Construção da visualização

A célula converte as métricas já calculadas em uma comparação visual. Nenhuma decisão adicional é tomada aqui; o gráfico apenas torna os resultados mais fáceis de inspecionar.


In [ ]:
rankings={'bm25_aliases':bm25_rankings,'multilingual_e5_small':dense_rankings,'bm25_e5_rrf':hybrid_rankings}
metrics={}; details={}
for name,value in rankings.items(): metrics[name],details[name]=evaluate(value)
best=max(metrics,key=lambda x:(metrics[x]['recall@5'],metrics[x]['mrr'],metrics[x]['ndcg@5']))
comparison={'bertimbau_embeddings_notebook09':previous['metrics']['bertimbau_embeddings'],**metrics}
names=list(comparison); keys=['recall@1','recall@3','recall@5','mrr','ndcg@5']; x=np.arange(len(keys)); width=.8/len(names)
fig,ax=plt.subplots(figsize=(12,6))
for i,name in enumerate(names):
    bars=ax.bar(x+(i-(len(names)-1)/2)*width,[comparison[name][k] for k in keys],width,label=name); ax.bar_label(bars,fmt='%.2f',padding=2,fontsize=8,rotation=90)
ax.set_xticks(x,keys); ax.set_ylim(0,1.13); ax.set_ylabel('Pontuação'); ax.set_title('Embeddings especializados versus baselines'); ax.grid(axis='y',alpha=.2); ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=2); fig.tight_layout(rect=(0,.08,1,1))
FIGURE_PATH.parent.mkdir(parents=True,exist_ok=True); fig.savefig(FIGURE_PATH,dpi=160,bbox_inches='tight'); plt.show()
report={'schema_version':'1.0','model':MODEL_NAME,'documents':len(kb),'evaluation_queries':len(queries),'selection_rule':'recall@5_then_mrr_then_ndcg@5','metrics':metrics,'notebook09_reference':previous['metrics'],'best_retriever':best,'query_details':details,'embedding':{'dimensions':int(doc_embeddings.shape[1]),'max_length':MAX_LENGTH,'device':str(DEVICE),'document_seconds':doc_seconds,'query_seconds':query_seconds},'warning':'Avaliação inicial curada a partir da própria base; requer revisão humana e novas consultas externas.'}
REPORT_PATH.parent.mkdir(parents=True,exist_ok=True); REPORT_PATH.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print({name:{k:round(v,4) for k,v in values.items()} for name,values in metrics.items()}); print({'best_retriever':best,'report':str(REPORT_PATH.relative_to(ROOT))})


## 5. Interpretação
O vencedor é escolhido automaticamente e será consumido pelo notebook 11. Uma melhora neste conjunto não equivale a qualidade em produção: precisamos aumentar as consultas, revisá-las com especialistas e medir respostas ponta a ponta.

A leitura dos resultados deve considerar tanto o padrão observado quanto as limitações da referência. As conclusões desta seção orientam a próxima etapa, mas não substituem a validação humana prevista no projeto.


### Limpeza de memória

No fluxo original, cada notebook começa com um kernel separado. Aqui removemos modelos e estruturas grandes que já foram persistidos em disco antes de continuar.


In [ ]:
# Libera objetos grandes antes da próxima etapa do notebook único.
import gc
for _variable in [
    'model', 'classifier', 'embedding_model', 'e5_model', 'tokenizer', 'e5_tokenizer',
    'knowledge_base', 'kb', 'chunks', 'meetings', 'document_embeddings',
    'doc_embeddings', 'query_embeddings', 'dense_score_matrix', 'train_loader',
    'validation_loader', 'train_dataset', 'validation_dataset'
]:
    globals().pop(_variable, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass


---

# Etapa 11 de 11 — 11 final integration

Esta seção reproduz a etapa `11_final_integration.ipynb` já reorganizada. As células devem ser executadas em ordem porque seus artefatos alimentam as etapas seguintes.


# 11 — Integração final: reunião → oportunidade → RAG fundamentado

Este notebook conecta o classificador BERTimbau aos documentos da base TOTVS. Ele produz um relatório estruturado por reunião com probabilidade, produtos/temas recuperados, evidências, fontes e políticas de uso.

**Privacidade:** nenhuma transcrição ou trecho é salvo nos relatórios. **Limite:** os rótulos de treinamento ainda são pseudo-rótulos; portanto o resultado é apoio à revisão comercial, não decisão automática.

**Objetivo e conexão com o projeto.** A etapa final conecta classificação, agregação por reunião e recuperação fundamentada para produzir insights estruturados sem armazenar o texto das transcrições. Os limiares são regras operacionais provisórias. A elevada presença de contexto misto é mantida como sinal de revisão e reforça a necessidade de calibração com rótulos humanos.


## 1. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [ ]:
from __future__ import annotations
import json, math, re, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
SEED=42; CHUNK_THRESHOLD=.80; MIN_SUPPORTING_CHUNKS=2; MIN_SUPPORT_DENSITY=.05; TOP_CHUNKS=3; TOP_DOCS=3; MAX_LENGTH=512
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); BATCH_SIZE=32 if DEVICE.type=='cuda' else 4


### Função auxiliar: `find_root`

Esta célula isola a responsabilidade implementada por `find_root`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def find_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/'data').exists() and (p/'notebooks').exists(): return p
    raise FileNotFoundError('Raiz do projeto não encontrada.')


### Preparação dos objetos desta etapa

A célula prepara `ROOT`, `CHUNKS_PATH`, `MODEL_PATH`, `KB_PATH` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
ROOT=find_root(); CHUNKS_PATH=ROOT/'data/processed/chunks_bertimbau.jsonl'; MODEL_PATH=ROOT/'data/processed/bertimbau_opportunity_best'
KB_PATH=ROOT/'data/knowledge_base/totvs_rag_kb_v1.json'; ALIASES_PATH=ROOT/'data/knowledge_base/rag_aliases.json'
RAG09_PATH=ROOT/'reports/metrics/rag_retrieval_evaluation.json'; RAG10_PATH=ROOT/'reports/metrics/sentence_embeddings_retrieval.json'
E5_EMBEDDINGS_PATH=ROOT/'data/processed/rag_multilingual_e5_small_embeddings.npz'; E5_MODEL_NAME='intfloat/multilingual-e5-small'
OUTPUT_PATH=ROOT/'data/processed/meeting_commercial_insights.jsonl'; REPORT_PATH=ROOT/'reports/metrics/final_integration_summary.json'
print({'device':str(DEVICE),'chunks_file':CHUNKS_PATH.exists(),'classifier':MODEL_PATH.exists()})


## 2. Classificação dos chunks
Todos os chunks são avaliados em lotes. Para reduzir falsos alarmes em reuniões longas, uma reunião candidata precisa de pelo menos dois chunks com probabilidade ≥ 0,80 e densidade mínima de 5%. Esses limiares são heurísticos até a auditoria humana. Guardamos somente índices, probabilidades e metadados não textuais.

O processamento em chunks resolve o limite de contexto sem perder a ligação com a reunião. Índices, offsets e sobreposição permitem reconstruir a ordem das evidências e agregar previsões posteriormente.


In [ ]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as f: return [json.loads(line) for line in f if line.strip()]


### Leitura ou escrita dos artefatos

A operação de arquivo fica isolada nesta célula para tornar claro quais dados entram ou saem da etapa e em que momento as validações são aplicadas.


In [ ]:
chunks=read_jsonl(CHUNKS_PATH); tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH); classifier=AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE).eval()
probabilities=[]; started=time.perf_counter()
for start in range(0,len(chunks),BATCH_SIZE):
    batch=[x['text'] for x in chunks[start:start+BATCH_SIZE]]
    encoded=tokenizer(batch,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda'):
        logits=classifier(**encoded).logits
    probabilities.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().tolist())
assert len(probabilities)==len(chunks); classification_seconds=time.perf_counter()-started
by_meeting=defaultdict(list)
for chunk,prob in zip(chunks,probabilities): by_meeting[str(chunk['meeting_id'])].append((float(prob),chunk))
print({'chunks':len(chunks),'meetings':len(by_meeting),'seconds':round(classification_seconds,2)})


## 3. Recuperação e políticas de grounding
O notebook lê e usa o vencedor do experimento 10. O E5 especializado usa o índice vetorial persistido; o BM25 permanece como fallback explícito para ambientes sem suporte ao vencedor.

A recuperação conecta sinais da reunião à base de conhecimento. Além da similaridade, cada resultado precisa conservar metadados de evidência e fonte para sustentar o uso responsável do contexto.


In [ ]:
kb=json.loads(KB_PATH.read_text(encoding='utf-8')); aliases=json.loads(ALIASES_PATH.read_text(encoding='utf-8'))['groups']; doc_ids=[d['id'] for d in kb]
rag09=json.loads(RAG09_PATH.read_text(encoding='utf-8')); rag10=json.loads(RAG10_PATH.read_text(encoding='utf-8'))
experiment_winner=rag10['best_retriever']; implemented_retriever=experiment_winner; fallback_used=False
STOP={'a','ao','aos','as','com','como','da','das','de','do','dos','e','em','entre','essa','esse','esta','este','foi','isso','mais','mas','muito','na','não','nas','no','nos','o','os','ou','para','pela','pelo','por','qual','que','se','sem','ser','sua','são','tem','um','uma','você','vs'}


### Função auxiliar: `norm`

Esta célula isola a responsabilidade implementada por `norm`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def norm(s):
    s=unicodedata.normalize('NFKD',str(s).casefold()); return ''.join(c for c in s if not unicodedata.combining(c))


### Preparação dos objetos desta etapa

A célula prepara `alias_variants` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
alias_variants=[[norm(x) for x in [g['canonical'],*g['aliases']]] for g in aliases]


### Funções auxiliares da seção

Esta célula agrupa `tokens`, `weighted`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def tokens(text,expand=True):
    n=norm(text); out=[x for x in re.findall(r'[a-z0-9+]+',n) if len(x)>=2 and x not in STOP]
    if expand:
        padded=f' {n} '
        for variants in alias_variants:
            if any(f' {v} ' in padded for v in variants): out.extend(tokens(' '.join(variants),False))
    return out
def weighted(d):
    fields=[(d.get('title',''),3),(d.get('product',''),3),(' '.join(d.get('keywords',[])),2),(d.get('category',''),1),(' '.join(d.get('segments',[])),1),(' '.join(d.get('related_products',[])),1),(' '.join(d.get('competitors',[])),2),(d.get('content',''),1)]
    return [t for value,w in fields for t in tokens(value)*w]


### Preparação dos objetos desta etapa

A célula prepara `doc_tokens`, `tf`, `lengths`, `avg` para a operação seguinte. O agrupamento mantém juntas apenas atribuições relacionadas e deixa a execução principal em uma célula própria.


In [ ]:
doc_tokens=[weighted(d) for d in kb]; tf=[Counter(x) for x in doc_tokens]; lengths=np.array([len(x) for x in doc_tokens],float); avg=lengths.mean(); df=Counter()
for x in doc_tokens: df.update(set(x))


### Função auxiliar: `bm25_scores`

Esta célula isola a responsabilidade implementada por `bm25_scores`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def bm25_scores(text):
    scores=np.zeros(len(kb))
    for term in tokens(text):
        n=df.get(term,0)
        if not n: continue
        idf=math.log(1+(len(kb)-n+.5)/(n+.5))
        for i,freqs in enumerate(tf):
            f=freqs.get(term,0)
            if f: scores[i]+=idf*f*2.5/(f+1.5*(.25+.75*lengths[i]/avg))
    return scores


### Verificações de integridade

As asserções tornam explícitas as condições que precisam permanecer verdadeiras para que os resultados sejam considerados consistentes.


In [ ]:
e5_tokenizer=e5_model=e5_doc_embeddings=None
if experiment_winner=='multilingual_e5_small':
    saved=np.load(E5_EMBEDDINGS_PATH); assert saved['document_ids'].tolist()==doc_ids
    e5_doc_embeddings=saved['embeddings']; e5_tokenizer=AutoTokenizer.from_pretrained(E5_MODEL_NAME); e5_model=AutoModel.from_pretrained(E5_MODEL_NAME).to(DEVICE).eval()
elif experiment_winner not in {'bm25_aliases'}:
    implemented_retriever='bm25_aliases'; fallback_used=True


### Funções auxiliares da seção

Esta célula agrupa `retrieve_many`, `policy`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def retrieve_many(texts,top_k=TOP_DOCS):
    if implemented_retriever=='multilingual_e5_small':
        vectors=[]
        for start in range(0,len(texts),BATCH_SIZE):
            batch=['query: '+x for x in texts[start:start+BATCH_SIZE]]; encoded=e5_tokenizer(batch,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
            with torch.inference_mode(), torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda'): hidden=e5_model(**encoded).last_hidden_state
            mask=encoded['attention_mask'].unsqueeze(-1); pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1); vectors.append(F.normalize(pooled,p=2,dim=1).cpu().numpy())
        score_matrix=np.concatenate(vectors)@e5_doc_embeddings.T
        return [[kb[i] for i in np.argsort(-scores,kind='stable')[:top_k]] for scores in score_matrix]
    return [[kb[i] for i in np.argsort(-bm25_scores(text),kind='stable')[:top_k]] for text in texts]
def policy(d):
    level=norm(d.get('evidence_level','')); sources=d.get('sources',[])
    if not sources: return 'context_only_do_not_state_as_fact_without_source'
    if 'hipotese' in level: return 'state_only_as_hypothesis_with_citation'
    if 'recomendacao' in level or 'validacao' in level: return 'state_as_recommendation_or_pending_validation_with_citation'
    return 'factual_claim_requires_citation'


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [ ]:
print({'experiment_winner':experiment_winner,'implemented_retriever':implemented_retriever,'fallback_used':fallback_used})


## 4. Agregação e insights por reunião
A decisão combina quantidade e densidade de chunks fortes; o máximo e a média dos três maiores scores ficam como indicadores auxiliares. Uma reunião candidata é marcada como contexto misto quando também possui ao menos dois chunks fortemente negativos. Documentos sem fonte podem ajudar na triagem, mas não geram afirmações factuais.


In [ ]:
top_by_meeting={meeting_id:sorted(items,key=lambda x:x[0],reverse=True)[:TOP_CHUNKS] for meeting_id,items in by_meeting.items()}


### Função auxiliar: `meeting_is_candidate`

Esta célula isola a responsabilidade implementada por `meeting_is_candidate`. A definição prepara essa operação para uso nas células seguintes e não altera os dados até ser chamada.


In [ ]:
def meeting_is_candidate(items):
    supporting=sum(prob>=CHUNK_THRESHOLD for prob,_ in items); return supporting>=MIN_SUPPORTING_CHUNKS and supporting/len(items)>=MIN_SUPPORT_DENSITY


### Leitura ou escrita dos artefatos

A operação de arquivo fica isolada nesta célula para tornar claro quais dados entram ou saem da etapa e em que momento as validações são aplicadas.


In [ ]:
retrieval_requests=[(meeting_id,chunk['chunk_index'],chunk['text']) for meeting_id,top in top_by_meeting.items() if meeting_is_candidate(by_meeting[meeting_id]) for _,chunk in top]
retrieval_batches=retrieve_many([x[2] for x in retrieval_requests]) if retrieval_requests else []
retrieval_lookup={(meeting_id,chunk_index):docs for (meeting_id,chunk_index,_),docs in zip(retrieval_requests,retrieval_batches)}
rows=[]
for meeting_id,items in by_meeting.items():
    top=top_by_meeting[meeting_id]; max_prob=top[0][0]; top_mean=float(np.mean([x[0] for x in top])); supporting=sum(prob>=CHUNK_THRESHOLD for prob,_ in items); density=supporting/len(items); positive=meeting_is_candidate(items)
    conflict=positive and sum(prob<=.20 for prob,_ in items)>=2
    retrieved=[]; seen=set()
    if positive:
        for prob,chunk in top:
            for d in retrieval_lookup[(meeting_id,chunk['chunk_index'])]:
                if d['id'] not in seen: retrieved.append(d); seen.add(d['id'])
    evidence=[]
    for d in retrieved[:5]:
        urls=[s['url'] for s in d.get('sources',[])]; p=policy(d)
        evidence.append({'document_id':d['id'],'title':d.get('title'),'product':d.get('product'),'category':d.get('category'),'evidence_level':d.get('evidence_level'),'source_urls':urls,'claim_policy':p,'can_support_fact':bool(urls) and p=='factual_claim_requires_citation'})
    factual=[e for e in evidence if e['can_support_fact']]; hypotheses=[e for e in evidence if e['claim_policy']=='state_only_as_hypothesis_with_citation']
    insight={'status':'oportunidade_para_revisao' if positive else 'sem_oportunidade_detectada','summary':('Possível oportunidade comercial detectada; revisar os produtos e evidências recuperados.' if positive else 'Nenhum chunk ultrapassou o limiar de oportunidade.'),'factual_documents':[e['document_id'] for e in factual],'hypothesis_documents':[e['document_id'] for e in hypotheses],'citation_required':bool(factual or hypotheses)}
    rows.append({'meeting_id':meeting_id,'opportunity_probability_max':round(max_prob,6),'opportunity_probability_top3_mean':round(top_mean,6),'high_opportunity_chunk_count':supporting,'high_opportunity_chunk_density':round(density,6),'opportunity_label':int(positive),'conflicting_chunks':conflict,'supporting_chunk_indices':[x[1]['chunk_index'] for x in top] if positive else [],'retriever':implemented_retriever,'retrieval_evidence':evidence,'commercial_insight':insight})
rows.sort(key=lambda x:(-x['opportunity_probability_max'],x['meeting_id']))
OUTPUT_PATH.parent.mkdir(parents=True,exist_ok=True)
with OUTPUT_PATH.open('w',encoding='utf-8') as f:
    for row in rows: f.write(json.dumps(row,ensure_ascii=False)+'\n')
print({'meetings_written':len(rows),'opportunities':sum(x['opportunity_label'] for x in rows),'conflicts':sum(x['conflicting_chunks'] for x in rows)})


## 5. Validações do contrato e relatório
Estas validações medem integridade, privacidade e grounding — não precisão de negócio. A avaliação ponta a ponta continuará pendente até existirem reuniões anotadas por humanos.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [ ]:
assert len(rows)==len(by_meeting)==len({x['meeting_id'] for x in rows})
assert all('text' not in row and 'transcription' not in row for row in rows)
assert all(e['source_urls'] or not e['can_support_fact'] for row in rows for e in row['retrieval_evidence'])
assert all(not (e['claim_policy']=='state_only_as_hypothesis_with_citation' and e['can_support_fact']) for row in rows for e in row['retrieval_evidence'])
opportunities=[x for x in rows if x['opportunity_label']==1]; grounded=[x for x in opportunities if any(e['source_urls'] for e in x['retrieval_evidence'])]
summary={'schema_version':'1.0','pipeline':f'BERTimbau opportunity classifier -> meeting aggregation -> {implemented_retriever} -> grounded structured insight','chunks':len(chunks),'meetings':len(rows),'opportunity_meetings':len(opportunities),'opportunity_rate':len(opportunities)/len(rows),'conflicting_meetings':sum(x['conflicting_chunks'] for x in rows),'grounded_opportunity_meetings':len(grounded),'grounded_coverage':len(grounded)/len(opportunities) if opportunities else 0,'thresholds':{'chunk_probability':CHUNK_THRESHOLD,'minimum_supporting_chunks':MIN_SUPPORTING_CHUNKS,'minimum_support_density':MIN_SUPPORT_DENSITY},'aggregation':{'decision':'minimum count and density of high-probability chunks','support':'maximum and mean of top 3 chunk probabilities','conflict':'candidate meeting with at least two chunks <= 0.20'},'retrieval':{'experiment_winner':experiment_winner,'implemented':implemented_retriever,'fallback_used':fallback_used},'contracts':{'transcript_text_not_saved':True,'sources_required_for_factual_claims':True,'hypotheses_never_marked_as_facts':True},'classification_seconds':classification_seconds,'warning':'Thresholds are operational heuristics. End-to-end metrics and production selection require human meeting labels.'}
REPORT_PATH.parent.mkdir(parents=True,exist_ok=True); REPORT_PATH.write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))

## 6. Próximos passos obrigatórios
1. Anotar a fila humana e criar teste final por reunião. 2. Revisar as 32 consultas do RAG com especialista e ampliar o conjunto. 3. Avaliar qualidade dos insights, correção das fontes e taxa de abstenção. 4. Somente então decidir limiar, agregação e modelo para produção.

Os itens a seguir separam o que já foi demonstrado do que ainda depende de dados humanos, calibração ou evolução técnica. Assim, o notebook termina sem transformar resultados experimentais em garantias de produção.


---

# Conclusão

Ao final da execução, os principais resultados ficam em:

- `reports/metrics/model_comparison.json`, para a comparação dos classificadores;
- `reports/metrics/sentence_embeddings_retrieval.json`, para a avaliação do RAG;
- `reports/metrics/final_integration_summary.json`, para o resumo da integração;
- `data/processed/meeting_commercial_insights.jsonl`, para os insights por reunião.

Os artefatos finais não armazenam o texto das transcrições. A leitura conjunta dos resultados mostra que o pipeline está integrado, mas a seleção definitiva e a calibração dos limiares ainda dependem da auditoria humana prevista no projeto.
